In [2]:
import qiime2 as q2
from qiime2 import Artifact
from qiime2 import Metadata
import pandas as pd
import matplotlib.pyplot as plt
import os
from qiime2.plugins.gemelli.actions import ctf, rpca
from qiime2.plugins.emperor.visualizers import biplot, plot
from qiime2.plugins.feature_table.methods import filter_samples

In [3]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
#from adjustText import adjust_text

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact

# Gemelli RPCA and rarefaction utilities
from gemelli.rpca import rpca # Optional: import auto_rpca if you use auto_rpca later
from gemelli.utils import qc_rarefaction
from gemelli.preprocessing import rclr_transformation


In [5]:
import qiime2 as q2
from qiime2 import Artifact
from qiime2 import Metadata
import pandas as pd
import matplotlib.pyplot as plt
import os
from qiime2.plugins.gemelli.actions import ctf, rpca
from qiime2.plugins.emperor.visualizers import biplot, plot
from qiime2.plugins.feature_table.methods import filter_samples

In [6]:
v3_table = Artifact.load('../data_Tyler/174950_rarefied_table.qza')
v4_table = Artifact.load('../data_Tyler/174951_rarefied_table.qza')
metadata = Metadata.load('../data_Tyler/57610_57610_analysis_mapping.txt')
taxonomy = Metadata.load('../data_Tyler/174116_taxonomy.tsv')

v3_table_filt = filter_samples(table=v3_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)
v4_table_filt = filter_samples(table=v4_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)

In [7]:
# first we import the metdata into pandas
mf = pd.read_csv('../data_Tyler/57610_57610_analysis_mapping.txt', sep='\t', index_col=0)
mf = mf.loc[mf.sample_type=='skin']
mf['acne_severity']= mf[['visual_assessment_on_photography_acne_severity_cheek_left', 'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['inflammatory_lesions_face']= mf[['visual_assessment_in_vivo_number_of_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['noninflammatory_lesions_face']= mf[['visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['a']= mf[['skincam_a_cheek_left','skincam_a_cheek_right']].replace('not applicable', 0).astype(float).sum(axis=1)

mf.loc[mf['subject_randomization_number'].astype(int)>299, 'cohort'] = 'acne'
mf.loc[mf['subject_randomization_number'].astype(int)<299, 'cohort'] = 'control'

mf['subject_randomization_id'] = mf['subject_randomization_number'].apply(lambda x: f"RD{int(x):03d}" if 0 <= int(x) <= 399 else x)

#mf=mf.drop(columns='Subject_Randomization_Number')
mf['class'] = mf['acne_severity'].apply(lambda x: 'acne' if x >= 1 else 'healthy')
mf['day'] = [int(day.replace('D', '')) if day != 'not applicable' else day for day in mf['day']]#mf['day'].str.replace('D', '').astype(int)

# Define the conditions for assigning values to the "category" column
condition1 = (mf['c_zone'] == 'C3')
condition2 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'acne'))
condition3 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'control'))

# Use the conditions to assign values to the "category" column
mf.loc[condition1, 'category'] = 'clear_zone'
mf.loc[condition2, 'category'] = 'acne'
mf.loc[condition3, 'category'] = 'control'

features_of_interest = ['a','skin_ph_meter_ph_mean_forehead_right', 'sebumeter_casual_level_mean_forehead_left',
                    'acne_severity', 'inflammatory_lesions_face', 'noninflammatory_lesions_face','day']#,'subject_randomization_id','subject_randomization_number', 'class', 'cohort', 'Current_Natural_Log_Ratio']

mf = mf.astype({col: 'str' for col in mf.columns[mf.dtypes == 'bool']})

In [8]:
print(mf)

                               processing_robot well_id_96          orig_name  \
#SampleID                                                                       
173620.14901.LAMI.RD308.D16.C1        Echo550_1         B4  LAMI.RD308.D16.C1   
173620.14901.LAMI.RD310.D21.C1        Echo550_1        A12  LAMI.RD310.D21.C1   
173620.14901.LAMI.RD305.D21.C3        Echo550_1        H10  LAMI.RD305.D21.C3   
173620.14901.LAMI.RD306.D18.C2        Echo550_1         D1  LAMI.RD306.D18.C2   
173620.14901.LAMI.RD306.D7.C2         Echo550_1        E10   LAMI.RD306.D7.C2   
...                                         ...        ...                ...   
173980.14901.LAMI.RD304.D28.C2        Echo550_1         G4  LAMI.RD304.D28.C2   
173980.14901.LAMI.RD313.D14.C1        Echo550_1         G5  LAMI.RD313.D14.C1   
173980.14901.LAMI.RD317.D9.C2         Echo550_1         F1   LAMI.RD317.D9.C2   
173980.14901.LAMI.RD319.D28.C1        Echo550_1         F9  LAMI.RD319.D28.C1   
173980.14901.LAMI.RD314.D25.

In [9]:
print(meta_v3)

NameError: name 'meta_v3' is not defined

In [10]:
# Create a new column 'category_label' for more descriptive labels
category_mapping = {
    'clear_zone': 'Acne Non-lesional',
    'acne': 'Acne Lesional',
    'control': 'Healthy'
}
mf['category_label'] = mf['category'].map(category_mapping)

# Use 'category_label' in metadata for the CTF analysis and plots
meta_v3 = mf.copy()
meta_v4 = mf.copy()

meta_v3['c_zone_group'] = meta_v3.cohort+'_'+meta_v3.c_zone
meta_v3['id_loc'] = meta_v3.subject_randomization_id+'_'+meta_v3.c_zone

meta_v4['c_zone_group'] = meta_v4.cohort+'_'+meta_v4.c_zone
meta_v4['id_loc'] = meta_v4.subject_randomization_id+'_'+meta_v4.c_zone

q2_meta_v3 = Metadata(meta_v3.applymap(lambda x: str(x) if isinstance(x, bool) else x))
q2_meta_v4 = Metadata(meta_v4.applymap(lambda x: str(x) if isinstance(x, bool) else x))

In [ ]:
# run CTF
ctf_results = ctf(v3_table_filt.filtered_table, q2_meta_v3,
                  'id_loc',
                  'day',
                  feature_metadata=taxonomy)
# expand the results
subject_biplot = ctf_results[0]
state_biplot = ctf_results[1]
distance_matrix = ctf_results[2]
state_subject_ordination = ctf_results[3]
state_feature_ordination = ctf_results[4]

In [11]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Calculate the angle between the x-axis and the largest eigenvector
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    
    # Width and height of the ellipse (scaling applied)
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    
    # Create the ellipse and add it to the plot
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

In [ ]:
# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_table_filt.filtered_table, q2_meta_v3, 'CTF - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/v1_v3'),
    (v4_table_filt.filtered_table, q2_meta_v4, 'CTF - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/v4')
]

# Update colors and group name mapping for new labels
group_colors = {
    "Healthy": "#423fa6",
    "Acne Lesional": "#df7963",
    "Acne Non-lesional": "#67b2be"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne Lesional": "Acne Lesional",
    "Acne Non-lesional": "Acne Non-lesional"
}

# Ensure output directories exist
for _, _, _, _, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)

# Function to extract last two parts of the taxon (if needed for plotting)
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]
    else:
        return ";".join(parts[-1:])


# Compositional Tensor Factorization

In [98]:
# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day', feature_metadata=taxonomy)
    
    # Unpack the results
    subject_biplot_artifact = ctf_results[0]  # This is a QIIME2 Artifact
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]  # This is also a QIIME2 Artifact
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract subject coordinates from the subject_biplot and name columns explicitly
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({'cohort':'first', 'class':'first', 'acne_severity':'first', 'c_zone':'first', 'category':'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)  # Apply category mapping for descriptive labels
    
    # Save metadata file in the specified output path
    mf.to_csv(os.path.join(output_path, 'subject-metadata.tsv'), sep='\t', index=True)
    
    # Add 'Group' column in spca_df based on 'category_label'
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a star at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    
    # Save the plot in the specified output path
    save_path = os.path.join(output_path, f"{file_suffix}_prevalence_unfiltered.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2422528061.py:63: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolo

Metadata saved to ../Figures/16S_Figures/CTF/v1_v3/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v1_v3/v1_v3_prevalence_unfiltered.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2422528061.py:63: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolo

Metadata saved to ../Figures/16S_Figures/CTF/v4/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v4/v4_prevalence_unfiltered.png


# CTF - Create subject distance matrix 

In [11]:
from skbio import OrdinationResults, DistanceMatrix
from scipy.spatial import distance
import qiime2 as q2
from qiime2.plugins.gemelli.actions import ctf

In [12]:
# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day', feature_metadata=taxonomy)
    
    # Unpack the results
    subject_biplot_artifact = ctf_results[0]  # This is a QIIME2 Artifact
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]  # This is also a QIIME2 Artifact
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

    # Create subject distance matrix
    subject_U = ctf_results.subject_biplot.view(OrdinationResults).samples.copy()
    Udist = distance.cdist(subject_U.copy(),
                        subject_U.copy())
    U_dist_res = DistanceMatrix(Udist, ids=subject_U.index)
    U_dist_res_q2 = q2.Artifact.import_data('DistanceMatrix', U_dist_res)
    # write the distances
    U_dist_res_q2.save(os.path.join(output_path, f"{file_suffix}_subject_distance.qza"))
    U_dist_res.write(os.path.join(output_path, f"{file_suffix}_subject_distance.txt"))
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract subject coordinates from the subject_biplot and name columns explicitly
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({'cohort':'first', 'class':'first', 'acne_severity':'first', 'c_zone':'first', 'category':'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)  # Apply category mapping for descriptive labels
    
    # Save metadata file in the specified output path
    mf.to_csv(os.path.join(output_path, 'subject-metadata.tsv'), sep='\t', index=True)
    
    # Add 'Group' column in spca_df based on 'category_label'
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a star at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    
    # Save the plot in the specified output path
    save_path = os.path.join(output_path, f"{file_suffix}_prevalence_unfiltered.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/v1_v3/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v1_v3/v1_v3_prevalence_unfiltered.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/v4/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v4/v4_prevalence_unfiltered.png


# CTF -- subsetted to the same number of samples and including the metabolomics 

In [61]:
from qiime2 import Artifact
import pandas as pd

# Step 1: Load the metabolomics table and extract sample names
data_table = Artifact.load('/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Output/data_sample_method2.qza')
metabolomics_sample_names = data_table.view(pd.DataFrame).index.tolist()
metabolomics_sample_names_set = set(metabolomics_sample_names)

# Step 2: Drop duplicates in metadata based on 'orig_name'
mf_unique = mf.drop_duplicates(subset=['orig_name'], keep='first')

# Step 3: Check if all metabolomics sample names are in 'mf_unique'
missing_samples = metabolomics_sample_names_set - set(mf_unique['orig_name'])

# Step 4: Raise a warning if any samples are missing
if missing_samples:
    print(f"Warning: The following samples are missing from the metadata: {missing_samples}")
else:
    print("All metabolomics sample names are present in the metadata.")




All metabolomics sample names are present in the metadata.


In [69]:
# Step 5: Subset metadata based on 'orig_name' in metabolomics sample names
mf_subset = mf_unique[mf_unique['orig_name'].isin(metabolomics_sample_names_set)]

# Step 6: Replace '#SampleID' with 'orig_name' and set it as the index
mf_subset = mf_subset.drop(columns=['#SampleID'], errors='ignore')  # Drop original if it exists
mf_subset = mf_subset.rename(columns={'orig_name': '#SampleID'})
mf_subset = mf_subset.set_index('#SampleID')

print(mf_subset)
# Step 7: Convert to QIIME 2 Metadata
mf_subset['c_zone_group'] = mf_subset.cohort+'_'+mf_subset.c_zone
mf_subset['id_loc'] = mf_subset.subject_randomization_id+'_'+mf_subset.c_zone

from qiime2 import Metadata

q2_meta_M = Metadata(mf_subset)

# Save to file (optional)
#q2_meta_M.save('filtered_metabolomics_metadata.tsv')

                  processing_robot well_id_96  platform  \
#SampleID                                                 
LAMI.RD310.D21.C1        Echo550_1        A12  Illumina   
LAMI.RD305.D21.C3        Echo550_1        H10  Illumina   
LAMI.RD306.D18.C2        Echo550_1         D1  Illumina   
LAMI.RD317.D14.C2        Echo550_1         F6  Illumina   
LAMI.RD305.D23.C1        Echo550_1         G2  Illumina   
...                            ...        ...       ...   
LAMI.RD305.D0.C2         Echo550_1         E7  Illumina   
LAMI.RD317.D21.C1        Echo550_1         E7  Illumina   
LAMI.RD001.D0.C1         Echo550_1         E8  Illumina   
LAMI.RD014.D14.C2        Echo550_1         H6  Illumina   
LAMI.RD001.D14.C2        Echo550_1         E4  Illumina   

                                                         pcr_primers  \
#SampleID                                                              
LAMI.RD310.D21.C1  FWD:GTGYCAGCMGCCGCGGTAA; REV:GGACTACNVGGGTWTCTAAT   
LAMI.RD305.D21.C

In [63]:
data_table=q2.Artifact.load('/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Output/data_sample_method2.qza')

In [64]:
data = data_table.view(pd.DataFrame)

In [65]:
print(data)

                         4293       15974        2531        5907  5946  \
LAMI.RD001.D0.C1   4770.76270      0.0000   72812.695  1022406.25   0.0   
LAMI.RD308.D2.C2   4166.87100      0.0000   56668.180   322180.03   0.0   
LAMI.RD304.D11.C1  2342.99900      0.0000  116903.530   548906.90   0.0   
LAMI.RD304.D11.C2   740.01514   6008.3945   69846.520   303948.28   0.0   
LAMI.RD304.D0.C1    725.83435   6228.5117   75098.500   438357.53   0.0   
...                       ...         ...         ...         ...   ...   
LAMI.RD318.D9.C3      0.00000      0.0000   53799.130   458503.40   0.0   
LAMI.RD308.D9.C3      0.00000      0.0000   30274.117  1192993.40   0.0   
LAMI.RD319.D2.C2   1567.89330      0.0000   63744.797   458884.62   0.0   
LAMI.RD319.D2.C3      0.00000  25945.3710   47938.910   403263.28   0.0   
LAMI.RD318.D21.C2     0.00000      0.0000   23423.326   415341.06   0.0   

                         2965      22668        1032         1074        955  \
LAMI.RD001.D0.C1   

In [18]:
data_v3 = v3_table_filt.filtered_table.view(pd.DataFrame)

In [19]:
print(data_v3)

                                GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT  \
173980.14901.LAMI.RD001.D0.C1                                                11.0                                                                                                        
173980.14901.LAMI.RD001.D0.C2                                                 7.0                                                                                                        
173980.14901.LAMI.RD001.D14.C1                                                0.0                                                                                                        
173980.14901.LAMI.RD001.D14.C2                                                3.0                                                                                                        
173980.14901.LAMI.RD001.D28.C1                                        

In [23]:
data_v4 = v4_table_filt.filtered_table.view(pd.DataFrame)
print(data_v4)

                                TACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTTAACTCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGG  \
173620.14901.LAMI.RD001.D0.C1                                                 0.0                                                                                                        
173620.14901.LAMI.RD001.D14.C1                                                0.0                                                                                                        
173620.14901.LAMI.RD004.D0.C2                                                 0.0                                                                                                        
173620.14901.LAMI.RD001.D0.C2                                                 0.0                                                                                                        
173620.14901.LAMI.RD004.D28.C1                                        

In [70]:
# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_table_filt.filtered_table, q2_meta_v3, 'CTF - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/v1_v3'),
    (v4_table_filt.filtered_table, q2_meta_v4, 'CTF - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/v4'), 
    (data_table, q2_meta_M, 'CTF - Metabolomics', 'metabolomics', '../Figures/16S_Figures/CTF/Metabolomics')
    
]

# Update colors and group name mapping for new labels
group_colors = {
    "Healthy": "#423fa6",
    "Acne Lesional": "#df7963",
    "Acne Non-lesional": "#67b2be"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne Lesional": "Acne Lesional",
    "Acne Non-lesional": "Acne Non-lesional"
}

# Ensure output directories exist
for _, _, _, _, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)


In [67]:
print(data_table)

<artifact: FeatureTable[Frequency] uuid: ceed42e1-6aa7-406e-b0a7-cb56070d4e76>


In [14]:
print(v3_table_filt.filtered_table)

<artifact: FeatureTable[Frequency] uuid: 48829c48-83cd-427a-a351-af82dedc31bd>


In [73]:
ctf_results = ctf(data_table, q2_meta_M, 'id_loc', 'day')

/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()


In [75]:
# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day')#, feature_metadata=taxonomy)
    
    # Unpack the results
    subject_biplot_artifact = ctf_results[0]  # This is a QIIME2 Artifact
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]  # This is also a QIIME2 Artifact
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

    # Create subject distance matrix
    subject_U = ctf_results.subject_biplot.view(OrdinationResults).samples.copy()
    Udist = distance.cdist(subject_U.copy(),
                        subject_U.copy())
    U_dist_res = DistanceMatrix(Udist, ids=subject_U.index)
    U_dist_res_q2 = q2.Artifact.import_data('DistanceMatrix', U_dist_res)
    # write the distances
    U_dist_res_q2.save(os.path.join(output_path, f"{file_suffix}_subject_distance.qza"))
    U_dist_res.write(os.path.join(output_path, f"{file_suffix}_subject_distance.txt"))
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract subject coordinates from the subject_biplot and name columns explicitly
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({'cohort':'first', 'class':'first', 'acne_severity':'first', 'c_zone':'first', 'category':'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)  # Apply category mapping for descriptive labels
    
    # Save metadata file in the specified output path
    mf.to_csv(os.path.join(output_path, 'subject-metadata.tsv'), sep='\t', index=True)
    
    # Add 'Group' column in spca_df based on 'category_label'
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a star at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    #plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    plt.legend(
        frameon=False, 
        fontsize=7, 
        title_fontsize=7, 
        loc="upper right",       # Set the position to the top-right corner
        bbox_to_anchor=(1.0, 1.0)  # Anchors the legend outside the plot area if necessary
    )
    # Save the plot in the specified output path
    save_path = os.path.join(output_path, f"{file_suffix}_prevalence_unfiltered.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/v1_v3/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v1_v3/v1_v3_prevalence_unfiltered.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/v4/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v4/v4_prevalence_unfiltered.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/Metabolomics/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/Metabolomics/metabolomics_prevalence_unfiltered.png


In [82]:
#Extract the number of similar ids between the tables 
from qiime2 import Artifact
import pandas as pd

# Load tables into pandas DataFrames
v3_samples = v3_table_filt.filtered_table.view(pd.DataFrame)
v4_samples = v4_table_filt.filtered_table.view(pd.DataFrame)
metabolomics_samples = data_table.view(pd.DataFrame)

# Clean sample names: split after the second dot to keep only the "LAMI" part
v3_samples_clean = v3_samples.rename(index=lambda x: x.split('.', 2)[-1])
v4_samples_clean = v4_samples.rename(index=lambda x: x.split('.', 2)[-1])

# Extract sets of cleaned sample names for comparison
v3_clean_ids = set(v3_samples_clean.index)
v4_clean_ids = set(v4_samples_clean.index)
metabolomics_ids = set(metabolomics_samples.index)


In [83]:
print(v3_samples_clean)

                   GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT  \
LAMI.RD001.D0.C1                                                11.0                                                                                                        
LAMI.RD001.D0.C2                                                 7.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD001.D14.C2                                                3.0                                                                                                        
LAMI.RD001.D28.C1                                                0.0                                                                   

In [102]:
# Find shared samples across all tables
shared_clean_ids = v3_clean_ids & v4_clean_ids & metabolomics_ids

# Print counts
print(f"Number of samples in V3 table: {len(v3_samples)}")
print(f"Number of samples in V4 table: {len(v4_samples)}")
print(f"Number of samples in Metabolomics table: {len(metabolomics_samples)}")
print(f"Number of shared samples across all tables: {len(shared_clean_ids)}")


Number of samples in V3 table: 237
Number of samples in V4 table: 218
Number of samples in Metabolomics table: 196
Number of shared samples across all tables: 148


In [103]:
print(shared_clean_ids)

{'LAMI.RD305.D21.C3', 'LAMI.RD305.D2.C2', 'LAMI.RD319.D28.C3', 'LAMI.RD001.D14.C1', 'LAMI.RD305.D28.C2', 'LAMI.RD305.D14.C1', 'LAMI.RD318.D28.C1', 'LAMI.RD306.D21.C2', 'LAMI.RD318.D11.C2', 'LAMI.RD317.D9.C2', 'LAMI.RD318.D14.C3', 'LAMI.RD319.D21.C2', 'LAMI.RD317.D0.C2', 'LAMI.RD319.D28.C2', 'LAMI.RD304.D0.C1', 'LAMI.RD305.D4.C2', 'LAMI.RD304.D4.C1', 'LAMI.RD310.D11.C1', 'LAMI.RD314.D7.C2', 'LAMI.RD317.D0.C3', 'LAMI.RD317.D7.C3', 'LAMI.RD310.D9.C1', 'LAMI.RD319.D2.C2', 'LAMI.RD310.D14.C2', 'LAMI.RD314.D25.C1', 'LAMI.RD317.D16.C2', 'LAMI.RD017.D28.C2', 'LAMI.RD318.D14.C1', 'LAMI.RD317.D14.C3', 'LAMI.RD314.D28.C1', 'LAMI.RD318.D9.C3', 'LAMI.RD308.D14.C2', 'LAMI.RD318.D0.C3', 'LAMI.RD304.D21.C2', 'LAMI.RD306.D28.C3', 'LAMI.RD313.D18.C1', 'LAMI.RD304.D18.C2', 'LAMI.RD314.D7.C3', 'LAMI.RD318.D2.C2', 'LAMI.RD313.D0.C1', 'LAMI.RD007.D28.C1', 'LAMI.RD313.D4.C2', 'LAMI.RD308.D9.C3', 'LAMI.RD310.D18.C1', 'LAMI.RD310.D21.C2', 'LAMI.RD313.D11.C1', 'LAMI.RD304.D11.C1', 'LAMI.RD318.D28.C2', 'LAMI.RD3

In [104]:
# Subset v3 and v4 using shared clean names
v3_subset = v3_samples[v3_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]
v4_subset = v4_samples[v4_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]

# Subset metabolomics table directly
metabolomics_subset = metabolomics_samples.loc[list(shared_clean_ids)]

# Confirm sizes
print(f"V3 table after subsetting: {v3_subset.shape}")
print(f"V4 table after subsetting: {v4_subset.shape}")
print(f"Metabolomics table after subsetting: {metabolomics_subset.shape}")


V3 table after subsetting: (148, 155)
V4 table after subsetting: (148, 218)
Metabolomics table after subsetting: (148, 1142)


In [105]:
# Convert back to QIIME 2 Artifacts
v3_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v3_subset)
v4_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v4_subset)
metabolomics_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', metabolomics_subset)


In [107]:
# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_subset_q2, q2_meta_v3, 'CTF - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/subset/v1_v3'),
    (v4_subset_q2, q2_meta_v4, 'CTF - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/subset/v4'), 
    (metabolomics_subset_q2, q2_meta_M, 'CTF - Metabolomics', 'metabolomics', '../Figures/16S_Figures/CTF/subset/Metabolomics')
]
# Update colors and group name mapping for new labels
group_colors = {
    "Healthy": "#423fa6",
    "Acne Lesional": "#df7963",
    "Acne Non-lesional": "#67b2be"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne Lesional": "Acne Lesional",
    "Acne Non-lesional": "Acne Non-lesional"
}

# Ensure output directories exist
for _, _, _, _, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)

In [108]:
# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day')#, feature_metadata=taxonomy)
    
    # Unpack the results
    subject_biplot_artifact = ctf_results[0]  # This is a QIIME2 Artifact
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]  # This is also a QIIME2 Artifact
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

    # Create subject distance matrix
    subject_U = ctf_results.subject_biplot.view(OrdinationResults).samples.copy()
    Udist = distance.cdist(subject_U.copy(),
                        subject_U.copy())
    U_dist_res = DistanceMatrix(Udist, ids=subject_U.index)
    U_dist_res_q2 = q2.Artifact.import_data('DistanceMatrix', U_dist_res)
    # write the distances
    U_dist_res_q2.save(os.path.join(output_path, f"{file_suffix}_subject_distance_subset.qza"))
    U_dist_res.write(os.path.join(output_path, f"{file_suffix}_subject_distance_subset.txt"))
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract subject coordinates from the subject_biplot and name columns explicitly
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({'cohort':'first', 'class':'first', 'acne_severity':'first', 'c_zone':'first', 'category':'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)  # Apply category mapping for descriptive labels
    
    # Save metadata file in the specified output path
    mf.to_csv(os.path.join(output_path, 'subject-metadata.tsv'), sep='\t', index=True)
    
    # Add 'Group' column in spca_df based on 'category_label'
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a star at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    #plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    plt.legend(
        frameon=False, 
        fontsize=7, 
        title_fontsize=7, 
        loc="upper right",       # Set the position to the top-right corner
        bbox_to_anchor=(1.0, 1.0)  # Anchors the legend outside the plot area if necessary
    )
    # Save the plot in the specified output path
    save_path = os.path.join(output_path, f"{file_suffix}_prevalence_unfiltered_subset.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset/v1_v3/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset/v1_v3/v1_v3_prevalence_unfiltered_subset.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset/v4/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset/v4/v4_prevalence_unfiltered_subset.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset/Metabolomics/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset/Metabolomics/metabolomics_prevalence_unfiltered_subset.png


In [109]:
# Find shared samples across all Metabolomics and V1-V3 tables
shared_clean_ids = v3_clean_ids & metabolomics_ids

# Print counts
print(f"Number of samples in V3 table: {len(v3_samples)}")
print(f"Number of samples in V4 table: {len(v4_samples)}")
print(f"Number of samples in Metabolomics table: {len(metabolomics_samples)}")
print(f"Number of shared samples across all tables: {len(shared_clean_ids)}")

Number of samples in V3 table: 237
Number of samples in V4 table: 218
Number of samples in Metabolomics table: 196
Number of shared samples across all tables: 175


In [110]:
# Subset v3 and v4 using shared clean names
v3_subset = v3_samples[v3_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]
v4_subset = v4_samples[v4_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]

# Subset metabolomics table directly
metabolomics_subset = metabolomics_samples.loc[list(shared_clean_ids)]

# Confirm sizes
print(f"V3 table after subsetting: {v3_subset.shape}")
print(f"V4 table after subsetting: {v4_subset.shape}")
print(f"Metabolomics table after subsetting: {metabolomics_subset.shape}")

V3 table after subsetting: (175, 155)
V4 table after subsetting: (148, 218)
Metabolomics table after subsetting: (175, 1142)


In [111]:
# Convert back to QIIME 2 Artifacts
v3_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v3_subset)
v4_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v4_subset)
metabolomics_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', metabolomics_subset)


In [ ]:
# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_subset_q2, q2_meta_v3, 'CTF - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/subset_Met_V1-V3/v1_v3'),
    (v4_subset_q2, q2_meta_v4, 'CTF - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/subset/v4'), 
    (metabolomics_subset_q2, q2_meta_M, 'CTF - Metabolomics', 'metabolomics', '../Figures/16S_Figures/CTF/subset_Met_V1-V3/Metabolomics')
]
# Update colors and group name mapping for new labels
group_colors = {
    "Healthy": "#423fa6",
    "Acne Lesional": "#df7963",
    "Acne Non-lesional": "#67b2be"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne Lesional": "Acne Lesional",
    "Acne Non-lesional": "Acne Non-lesional"
}

# Ensure output directories exist
for _, _, _, _, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)

In [113]:
# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day')#, feature_metadata=taxonomy)
    
    # Unpack the results
    subject_biplot_artifact = ctf_results[0]  # This is a QIIME2 Artifact
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]  # This is also a QIIME2 Artifact
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

    # Create subject distance matrix
    subject_U = ctf_results.subject_biplot.view(OrdinationResults).samples.copy()
    Udist = distance.cdist(subject_U.copy(),
                        subject_U.copy())
    U_dist_res = DistanceMatrix(Udist, ids=subject_U.index)
    U_dist_res_q2 = q2.Artifact.import_data('DistanceMatrix', U_dist_res)
    # write the distances
    U_dist_res_q2.save(os.path.join(output_path, f"{file_suffix}_subject_distance_subset_Met_V1-V3.qza"))
    U_dist_res.write(os.path.join(output_path, f"{file_suffix}_subject_distance_subset_Met_V1-V3.txt"))
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract subject coordinates from the subject_biplot and name columns explicitly
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({'cohort':'first', 'class':'first', 'acne_severity':'first', 'c_zone':'first', 'category':'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)  # Apply category mapping for descriptive labels
    
    # Save metadata file in the specified output path
    mf.to_csv(os.path.join(output_path, 'subject-metadata.tsv'), sep='\t', index=True)
    
    # Add 'Group' column in spca_df based on 'category_label'
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a star at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    #plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    plt.legend(
        frameon=False, 
        fontsize=7, 
        title_fontsize=7, 
        loc="upper right",       # Set the position to the top-right corner
        bbox_to_anchor=(1.0, 1.0)  # Anchors the legend outside the plot area if necessary
    )
    # Save the plot in the specified output path
    save_path = os.path.join(output_path, f"{file_suffix}_prevalence_unfiltered_subset_Met_V1-V3.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset_Met_V1-V3/v1_v3/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset_Met_V1-V3/v1_v3/v1_v3_prevalence_unfiltered_subset_Met_V1-V3.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset_Met_V1-V3/Metabolomics/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset_Met_V1-V3/Metabolomics/metabolomics_prevalence_unfiltered_subset_Met_V1-V3.png


In [123]:
#V4 and Metabolomics
# Find shared samples across all Metabolomics and V1-V3 tables
shared_clean_ids = v4_clean_ids & metabolomics_ids

# Print counts
print(f"Number of samples in V3 table: {len(v3_samples)}")
print(f"Number of samples in V4 table: {len(v4_samples)}")
print(f"Number of samples in Metabolomics table: {len(metabolomics_samples)}")
print(f"Number of shared samples across all tables: {len(shared_clean_ids)}")

Number of samples in V3 table: 237
Number of samples in V4 table: 218
Number of samples in Metabolomics table: 196
Number of shared samples across all tables: 163


In [126]:
# Subset v3 and v4 using shared clean names
v3_subset = v3_samples[v3_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]
v4_subset = v4_samples[v4_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]

# Subset metabolomics table directly
metabolomics_subset = metabolomics_samples.loc[list(shared_clean_ids)]

# Confirm sizes
print(f"V3 table after subsetting: {v3_subset.shape}")
print(f"V4 table after subsetting: {v4_subset.shape}")
print(f"Metabolomics table after subsetting: {metabolomics_subset.shape}")

V3 table after subsetting: (148, 155)
V4 table after subsetting: (163, 218)
Metabolomics table after subsetting: (163, 1142)


In [116]:
# Convert back to QIIME 2 Artifacts
v3_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v3_subset)
v4_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v4_subset)
metabolomics_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', metabolomics_subset)


In [117]:
# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    #(v3_subset_q2, q2_meta_v3, 'CTF - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/subset_Met_V4/v4'),
    (v4_subset_q2, q2_meta_v4, 'CTF - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/subset_Met_V4/v4'), 
    (metabolomics_subset_q2, q2_meta_M, 'CTF - Metabolomics', 'metabolomics', '../Figures/16S_Figures/CTF/subset_Met_V4/Metabolomics')
]
# Update colors and group name mapping for new labels
group_colors = {
    "Healthy": "#423fa6",
    "Acne Lesional": "#df7963",
    "Acne Non-lesional": "#67b2be"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne Lesional": "Acne Lesional",
    "Acne Non-lesional": "Acne Non-lesional"
}

# Ensure output directories exist
for _, _, _, _, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)

In [118]:
# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day')#, feature_metadata=taxonomy)
    
    # Unpack the results
    subject_biplot_artifact = ctf_results[0]  # This is a QIIME2 Artifact
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]  # This is also a QIIME2 Artifact
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

    # Create subject distance matrix
    subject_U = ctf_results.subject_biplot.view(OrdinationResults).samples.copy()
    Udist = distance.cdist(subject_U.copy(),
                        subject_U.copy())
    U_dist_res = DistanceMatrix(Udist, ids=subject_U.index)
    U_dist_res_q2 = q2.Artifact.import_data('DistanceMatrix', U_dist_res)
    # write the distances
    U_dist_res_q2.save(os.path.join(output_path, f"{file_suffix}_subject_distance_subset_Met_V4.qza"))
    U_dist_res.write(os.path.join(output_path, f"{file_suffix}_subject_distance_subset_Met_V4.txt"))
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract subject coordinates from the subject_biplot and name columns explicitly
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({'cohort':'first', 'class':'first', 'acne_severity':'first', 'c_zone':'first', 'category':'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)  # Apply category mapping for descriptive labels
    
    # Save metadata file in the specified output path
    mf.to_csv(os.path.join(output_path, 'subject-metadata.tsv'), sep='\t', index=True)
    
    # Add 'Group' column in spca_df based on 'category_label'
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a star at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    #plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    plt.legend(
        frameon=False, 
        fontsize=7, 
        title_fontsize=7, 
        loc="upper right",       # Set the position to the top-right corner
        bbox_to_anchor=(1.0, 1.0)  # Anchors the legend outside the plot area if necessary
    )
    # Save the plot in the specified output path
    save_path = os.path.join(output_path, f"{file_suffix}_prevalence_unfiltered_subset_Met_V4.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset_Met_V4/v4/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset_Met_V4/v4/v4_prevalence_unfiltered_subset_Met_V4.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset_Met_V4/Metabolomics/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset_Met_V4/Metabolomics/metabolomics_prevalence_unfiltered_subset_Met_V4.png


In [133]:
#V4 and V1_V3
# Find shared samples across all Metabolomics and V1-V3 tables
shared_clean_ids = v4_clean_ids & v3_clean_ids

# Print counts
print(f"Number of samples in V3 table: {len(v3_samples)}")
print(f"Number of samples in V4 table: {len(v4_samples)}")
print(f"Number of samples in Metabolomics table: {len(metabolomics_samples)}")
print(f"Number of shared samples across all tables: {len(shared_clean_ids)}")

Number of samples in V3 table: 237
Number of samples in V4 table: 218
Number of samples in Metabolomics table: 196
Number of shared samples across all tables: 197


In [138]:
# Subset v3 and v4 using shared clean names
v3_subset = v3_samples[v3_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]
v4_subset = v4_samples[v4_samples.index.map(lambda x: x.split('.', 2)[-1]).isin(shared_clean_ids)]



# Confirm sizes
print(f"V3 table after subsetting: {v3_subset.shape}")
print(f"V4 table after subsetting: {v4_subset.shape}")


V3 table after subsetting: (197, 155)
V4 table after subsetting: (197, 218)


In [139]:
# Convert back to QIIME 2 Artifacts
v3_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v3_subset)
v4_subset_q2 = Artifact.import_data('FeatureTable[Frequency]', v4_subset)


In [140]:
# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_subset_q2, q2_meta_v3, 'CTF - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/subset_V1V3_V4/v3'),
    (v4_subset_q2, q2_meta_v4, 'CTF - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/subset_V1V3_V4/v4')
    ]
# Update colors and group name mapping for new labels
group_colors = {
    "Healthy": "#423fa6",
    "Acne Lesional": "#df7963",
    "Acne Non-lesional": "#67b2be"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne Lesional": "Acne Lesional",
    "Acne Non-lesional": "Acne Non-lesional"
}

# Ensure output directories exist
for _, _, _, _, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)

In [141]:
# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day')#, feature_metadata=taxonomy)
    
    # Unpack the results
    subject_biplot_artifact = ctf_results[0]  # This is a QIIME2 Artifact
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]  # This is also a QIIME2 Artifact
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)

    # Convert the skbio DistanceMatrix to a Pandas DataFrame
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

    # Create subject distance matrix
    subject_U = ctf_results.subject_biplot.view(OrdinationResults).samples.copy()
    Udist = distance.cdist(subject_U.copy(),
                        subject_U.copy())
    U_dist_res = DistanceMatrix(Udist, ids=subject_U.index)
    U_dist_res_q2 = q2.Artifact.import_data('DistanceMatrix', U_dist_res)
    # write the distances
    U_dist_res_q2.save(os.path.join(output_path, f"{file_suffix}_subject_distance_subset_V1V3_V4.qza"))
    U_dist_res.write(os.path.join(output_path, f"{file_suffix}_subject_distance_subset_V1V3_V4.txt"))
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract subject coordinates from the subject_biplot and name columns explicitly
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({'cohort':'first', 'class':'first', 'acne_severity':'first', 'c_zone':'first', 'category':'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)  # Apply category mapping for descriptive labels
    
    # Save metadata file in the specified output path
    mf.to_csv(os.path.join(output_path, 'subject-metadata.tsv'), sep='\t', index=True)
    
    # Add 'Group' column in spca_df based on 'category_label'
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],
            edgecolor=group_colors[group_name],
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping[group_name]
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)

        # Draw the ellipse for the group
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

        # Add a star at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    #plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    plt.legend(
        frameon=False, 
        fontsize=7, 
        title_fontsize=7, 
        loc="upper right",       # Set the position to the top-right corner
        bbox_to_anchor=(1.0, 1.0)  # Anchors the legend outside the plot area if necessary
    )
    # Save the plot in the specified output path
    save_path = os.path.join(output_path, f"{file_suffix}_prevalence_unfiltered_subset_V1V3_V4.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset_V1V3_V4/v3/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset_V1V3_V4/v3/v1_v3_prevalence_unfiltered_subset_V1V3_V4.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_47284/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var

Metadata saved to ../Figures/16S_Figures/CTF/subset_V1V3_V4/v4/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/subset_V1V3_V4/v4/v4_prevalence_unfiltered_subset_V1V3_V4.png


# CTF with Features

In [70]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from qiime2 import Artifact
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from biom import load_table

# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))

# Helper function to extract last taxa
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    return ";".join(parts[-2:]) if "uncultured" not in taxon else parts[-3]


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()


NameError: name 'biom_path' is not defined

In [76]:
state_feature_ordination = ctf_results[4]

In [77]:
print(state_feature_ordination)

<artifact: FeatureData[FeatureTrajectory] uuid: 10bec35c-e5ae-4f22-8a7b-a1627f759f52>


In [78]:
# Convert `state_feature_ordination` to a DataFrame
feature_coordinates_df = state_feature_ordination.view(pd.DataFrame)
    

In [79]:
print(feature_coordinates_df)

                PC1       PC2       PC3  \
featureid                                 
0         -0.132651 -0.240360 -0.055020   
1         -0.132607 -0.183591 -0.001781   
2         -0.126138 -0.129051  0.053440   
3         -0.122882 -0.219963  0.022332   
4         -0.122437 -0.244087  0.016068   
...             ...       ...       ...   
2010      -0.010991  0.038282 -0.005477   
2011      -0.010269  0.037140 -0.005982   
2012      -0.010789  0.038016 -0.005912   
2013      -0.010714  0.038727 -0.005911   
2014      -0.010472  0.034585 -0.005886   

                                                  feature_id   day  \
featureid                                                            
0          GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAG...  25.0   
1          GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAG...   2.0   
2          GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAG...  23.0   
3          GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAG...  11.0   
4          GACGAACGC

In [88]:
#taxonomy = Artifact.load('../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
print(taxonomy)

                                                                                                Taxon  \
Feature ID                                                                                              
AACAAACGCTGGCGGCGCGTCTTAAGCATGCAAGTCGAACGGCAAGA...  d__Bacteria; p__Spirochaetota; c__Spirochaetia...   
AACAAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAACGGGGAAA...  d__Bacteria; p__Bdellovibrionota; c__Bdellovib...   
AACAAACGCTGGCGGCGTGCTTCATACATGCAAGTCGGACGAGAAAG...                                        d__Bacteria   
AACAGAGGATGCAAGCGTTATCCGGAATTATTGGGCGTAAAGTGTCT...  d__Bacteria; p__Cyanobacteria; c__Cyanobacteri...   
AACAGAGGATGCAAGCGTTATCCGGAATTATTGGGCGTAAAGTGTCT...  d__Bacteria; p__Cyanobacteria; c__Cyanobacteri...   
...                                                                                               ...   
TGCGTATGTCGCGAGCGTTATCCGGAATTATTGGGCATAAAGGGCAT...  d__Bacteria; p__Fusobacteriota; c__Fusobacteri...   
TTCGTAAGGACCGAGCGTTGTCCGGAATCATTGGGCGTAAAGGGTAC...  d__

In [81]:
print(prevalence)

GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT    0.236287
AGCGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAGCGCCGTAGCAATACGGAGCGGCAGACGGGTGAGTAACACGTGGGAACGTACCTTTTGGTTCGGAACAACACAGGGAAACTTGTGCTAATACCGGATAAGCCCTTACGGGGA    0.042194
ATTGAACGCTGGCGGCATGCCTTACACATGCAAGTCGAACGGTAACAGGTCTTCGGATGCTGACGAGTGGCGAACGGGTGAGTAATACATCGGAACGTGCCCGATCGTGGGGGATAACGGAGCGAAAGCTTTGCTAATACCGCATACGAT    0.054852
GATGAACGCTAGCGATAGGCTTAACACATGCAAGTCGAGGGGCAGCATGGTCTTAGCTTGCTAAGACTGATGGCGACCGGCGCACGGGTGCGTAACGCGTATGCAACTTGCCTCACAGAGGGGGATAACCCGTCGAAAGACGGACTAATA    0.261603
GACGAACGCTGGCGGCGCGCCTAACACATGCAAGTCGAACGGAGCTTAGAGAGCTTGCTTTTTAAGCTTAGTGGCGAACGGGTGAGTAACGCGTGAGTAACCTGCCCTAGAGTGGGGGACAACAGTTGGAAACGACTGCTAATACCGCAT    0.016878
                                                                                                                                                            ...   
GATGAACGCTGGCGGCGTGCTT

In [103]:
print(top_feature_coords)

                                                         PC1       PC2  \
feature_id                                                               
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132651 -0.240360   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132607 -0.183591   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.126138 -0.129051   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122882 -0.219963   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122437 -0.244087   
...                                                      ...       ...   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.124660 -0.243932   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.131331 -0.238916   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.126522 -0.242766   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.127214 -0.245889   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.129456 -0.227688   

                                     

In [ ]:
top_feature_coords['PC1'] = top_feature_coords['PC1'].astype(float)
top_feature_coords['PC2'] = top_feature_coords['PC2'].astype(float)


In [107]:
# Ensure PC1 and PC2 columns in `top_feature_coords` are fully numeric
top_feature_coords['PC1'] = pd.to_numeric(top_feature_coords['PC1'], errors='coerce')
top_feature_coords['PC2'] = pd.to_numeric(top_feature_coords['PC2'], errors='coerce')

# Drop any rows with NaN values in PC1 or PC2 after conversion
top_feature_coords = top_feature_coords.dropna(subset=['PC1', 'PC2'])

# Debugging output: Verify data types and contents of `top_feature_coords`
print("Top Feature Coordinates Data Types:\n", top_feature_coords.dtypes)
print("Top Feature Coordinates Preview:\n", top_feature_coords.head())


Top Feature Coordinates Data Types:
 PC1           float64
PC2           float64
PC3           float64
day           float64
Taxon          object
kingdom        object
phylum         object
class          object
order          object
family         object
genus          object
species        object
Confidence    float64
dtype: object
Top Feature Coordinates Preview:
                                                          PC1       PC2  \
feature_id                                                               
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132651 -0.240360   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132607 -0.183591   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.126138 -0.129051   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122882 -0.219963   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122437 -0.244087   

                                                         PC3   day  \
feature_id                              

In [110]:
print(prevalent_features)

Index(['GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT',
       'GATGAACGCTAGCGATAGGCTTAACACATGCAAGTCGAGGGGCAGCATGGTCTTAGCTTGCTAAGACTGATGGCGACCGGCGCACGGGTGCGTAACGCGTATGCAACTTGCCTCACAGAGGGGGATAACCCGTCGAAAGACGGACTAATA',
       'ATTGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAACGATGATTATCTAGCTTGCTAGATATGATTAGTGGCGGACGGGTGAGTAACATTTAGGAATCTGCCTAGTAGTGGGGGATAGCTCGGGGAAACTCGAATTAATACCGCATA',
       'ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGGACGGCAGCACAGAGAAGCTTGCTTCTTGGGTGGCGAGTGGCGAACGGGTGAGTAATATATCGGAACGTACCGAGTAATGGGGGATAACTAATCGAAAGATTAGCTAATACCG',
       'GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT',
       'GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGAAGAGCGACGGAAGCTTGCTTCTATCAATCTTAGTGGCGAACGGGTGAGTAACGCGTAATCAACCTGCCCTTCAGAGGGGGACAACAGTTGGAAACGACTGCTAATACC',
       'GATGAACGCTGGCGGCGTGCCTAATA

In [113]:
    # Convert `state_feature_ordination` to a DataFrame and use `feature_id` for indexing
feature_coordinates_df = state_feature_ordination.view(pd.DataFrame).set_index("feature_id")
print(feature_coordinates_df)

    # Filter feature coordinates by prevalent features and select top 10 by vector magnitude
feature_coordinates = feature_coordinates_df.loc[feature_coordinates_df.index.isin(prevalent_features)]
print(feature_coordinates)
top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
top_feature_coords = feature_coordinates.loc[top_features]

                                                         PC1       PC2  \
feature_id                                                               
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132651 -0.240360   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132607 -0.183591   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.126138 -0.129051   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122882 -0.219963   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122437 -0.244087   
...                                                      ...       ...   
GACGAACGCTGGCGGCACGCTTAACACATGCAAGTCGAACGAGAGAA... -0.010991  0.038282   
GACGAACGCTGGCGGCACGCTTAACACATGCAAGTCGAACGAGAGAA... -0.010269  0.037140   
GACGAACGCTGGCGGCACGCTTAACACATGCAAGTCGAACGAGAGAA... -0.010789  0.038016   
GACGAACGCTGGCGGCACGCTTAACACATGCAAGTCGAACGAGAGAA... -0.010714  0.038727   
GACGAACGCTGGCGGCACGCTTAACACATGCAAGTCGAACGAGAGAA... -0.010472  0.034585   

                                     

In [114]:
print(top_features)

Index(['GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT',
       'GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT',
       'GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT',
       'GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT',
       'GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT',
       'GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT',
       'GACGAACGCTGGCGGCGTGCTTAACA

In [115]:
print(top_feature_coords)

                                                         PC1       PC2  \
feature_id                                                               
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132651 -0.240360   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.132607 -0.183591   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.126138 -0.129051   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122882 -0.219963   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.122437 -0.244087   
...                                                      ...       ...   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.124660 -0.243932   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.131331 -0.238916   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.126522 -0.242766   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.127214 -0.245889   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG... -0.129456 -0.227688   

                                     

In [116]:
taxon = top_feature_coords.loc[feature, 'Taxon']

In [117]:
print(taxon)

feature_id
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT    d__Bacteria; p__Actinobacteriota; c__Actinobac...
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT    d__Bacteria; p__Actinobacteriota; c__Actinobac...
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT    d__Bacteria; p__Actinobacteriota; c__Actinobac...
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT    d__Bacteria; p__Actinobacteriota; c__Actinobac...
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT    d__Bacteria; p__Act

In [120]:
pc1, pc2 = top_feature_coords.loc[feature, ['PC1', 'PC2']]

In [125]:
    # Extract pc1 and pc2 as float values
pc1 = float(top_feature_coords.loc[feature, 'PC1'].iloc[0])
pc2 = float(top_feature_coords.loc[feature, 'PC2'].iloc[0])
#taxon = top_feature_coords.loc[feature, 'Taxon']

In [128]:
taxon = top_feature_coords.loc[feature, 'Taxon']

In [129]:
print(taxon)

feature_id
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT     g__Cutibacterium
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT     g__Cutibacterium
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT     g__Cutibacterium
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT     g__Cutibacterium
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCCTGCTTTTGTGGGGTGCTCGAGTGGCGAACGGGTGAGTAACACGTGAGTAACCTGCCCTTGACTTTGGGATAACTTCAGGAAACTGGGGCTAATACCGGAT     g__Cutibacterium
                                                                                                                                 

In [130]:
import os
import numpy as np
import pandas as pd
from biom import Table
from skbio.stats.ordination import OrdinationResults
from skbio import DistanceMatrix
import seaborn as sns
import matplotlib.pyplot as plt
from qiime2 import Artifact

# Main code
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF with taxonomy metadata
    ctf_results = ctf(table, metadata, 'id_loc', 'day', feature_metadata=taxonomy)
    subject_biplot_artifact = ctf_results[0]
    state_biplot = ctf_results[1]
    distance_matrix_artifact = ctf_results[2]
    state_subject_ordination = ctf_results[3]
    state_feature_ordination = ctf_results[4]

    # Convert distance matrix artifact to a skbio DistanceMatrix object
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
    # Convert subject biplot artifact to an OrdinationResults object
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']

    # Explicitly ensure PC1 and PC2 are numeric
    spca_df['PC1'] = pd.to_numeric(spca_df['PC1'], errors='coerce')
    spca_df['PC2'] = pd.to_numeric(spca_df['PC2'], errors='coerce')

    # Map 'category_label' from metadata to spca_df for the group column
    mf = metadata.to_dataframe().reset_index().groupby('id_loc').agg({
        'cohort': 'first', 'class': 'first', 'acne_severity': 'first', 'c_zone': 'first', 'category': 'first'})
    mf.index.name = '#SampleID'
    mf['category_label'] = mf['category'].map(category_mapping)
    spca_df["Group"] = spca_df.index.map(mf["category_label"])

    # Convert the QIIME2 Artifact (table) to a BIOM Table
    biom_table = table.view(Table)
    
    # Calculate prevalence and filter features
    prevalence = calculate_prevalence(biom_table)
    prevalent_features = prevalence[prevalence >= 0.1].index  # Keep features with >=10% prevalence

    # Convert `state_feature_ordination` to a DataFrame and use `feature_id` for indexing
    feature_coordinates_df = state_feature_ordination.view(pd.DataFrame).set_index("feature_id")
    
    # Filter feature coordinates by prevalent features and select top 10 by vector magnitude
    feature_coordinates = feature_coordinates_df.loc[feature_coordinates_df.index.isin(prevalent_features)]
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    top_feature_coords = feature_coordinates.loc[top_features]
    
    # Ensure that top feature coordinates are numeric
    top_feature_coords['PC1'] = pd.to_numeric(top_feature_coords['PC1'], errors='coerce')
    top_feature_coords['PC2'] = pd.to_numeric(top_feature_coords['PC2'], errors='coerce')

    # Convert taxonomy Metadata to a DataFrame and map feature IDs to taxonomy
    taxonomy_df = taxonomy.to_dataframe()
    top_feature_coords['Taxon'] = top_feature_coords.index.map(lambda fid: extract_last_two_taxa(taxonomy_df.loc[fid, 'Taxon']))

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for group_name, group in spca_df.groupby('Group'):
        # Plot group points
        sns.scatterplot(
            x=group["PC1"].values,
            y=group["PC2"].values,
            color=group_colors.get(group_name, 'grey'),  # Use default color if group not found
            alpha=0.8,
            linewidth=0.5,
            ax=ax,
            label=group_name_mapping.get(group_name, group_name)
        )

        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
        draw_ellipse(mean, cov, ax, group_colors.get(group_name, 'grey'), scale_factor=2)
        ax.scatter(mean[0], mean[1], color=group_colors.get(group_name, 'grey'), marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Feature arrows for top prevalent features
        # Feature arrows for top prevalent featu

    for feature in top_feature_coords.index:
        pc1, pc2 = float(top_feature_coords.loc[feature, ['PC1', 'PC2']].iloc[0])
        taxon = top_feature_coords.loc[feature, 'Taxon']
        
        # Ensure pc1 and pc2 are valid numeric values
        if not np.isnan(pc1) and not np.isnan(pc2):
            ax.arrow(0, 0, pc1, pc2, color='grey', alpha=0.3, head_width=0.02, length_includes_head=True)
            ax.text(pc1 * 1.1, pc2 * 1.1, taxon, fontsize=3, color='black', ha="center")
    
    # Customize plot labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    plt.legend(frameon=False, fontsize=7, title_fontsize=7)
    
    # Save plot
    save_path = os.path.join(output_path, f"{file_suffix}_top10_features_prevalence_filtered.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)
    
    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/953461292.py:85: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor

TypeError: cannot convert the series to <class 'float'>

#trying to adapt this to add local lesion severity score on the CTF plot

In [ ]:
# Group by 'subject_ID_CC' and calculate the mean of 'local_lesion_severity'
mean_severity = metadata.groupby('id_loc')['local_lesion_severity'].transform('mean')

# Add this mean value as a new column in metadata_df
metadata['local_lesion_severity_mean_ID_CC'] = mean_severity

print(metadata)

In [27]:
# Severity group 
metadata_df = pd.read_csv(f'../Metadata/metadata_final_22102024.tsv', sep='\t')



metadata_df['id_loc'] = metadata_df.subject_randomization_id+'_'+metadata_df.c_zone
# Group by 'subject_ID_CC' and calculate the mean of 'local_lesion_severity'
mean_severity = metadata_df.groupby('id_loc')['local_lesion_severity'].transform('mean')

# Add this mean value as a new column in metadata_df
metadata_df['local_lesion_severity_mean_ID_CC'] = mean_severity

In [28]:
print(metadata_df['id_loc'])

0      RD308_C1
1      RD310_C1
2      RD305_C3
3      RD306_C2
4      RD306_C2
         ...   
261    RD317_C1
262    RD001_C1
263    RD014_C2
264    RD314_C1
265    RD001_C2
Name: id_loc, Length: 266, dtype: object


In [29]:
mf = metadata_df.groupby('id_loc').agg({'group':'first','local_lesion_severity_mean_ID_CC':'first'})

In [32]:
print(mf)

             group  local_lesion_severity_mean_ID_CC
#SampleID                                           
RD001_C1   Healthy                          0.000000
RD001_C2   Healthy                          0.000000
RD002_C1   Healthy                          0.000000
RD002_C2   Healthy                          0.000000
RD003_C1   Healthy                          0.000000
RD003_C2   Healthy                          0.000000
RD004_C1   Healthy                          0.000000
RD004_C2   Healthy                          0.000000
RD006_C1   Healthy                          0.000000
RD006_C2   Healthy                          0.000000
RD007_C1   Healthy                          0.000000
RD007_C2   Healthy                          0.000000
RD011_C1   Healthy                          0.000000
RD011_C2   Healthy                          0.000000
RD014_C1   Healthy                          0.000000
RD014_C2   Healthy                          0.000000
RD017_C1   Healthy                          0.

In [31]:
mf.index.name = '#SampleID'

In [33]:
mf.to_csv('/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Figures/16S_Figures/CTF/subject-metadata_subject_ID_mean_severity.tsv', sep='\t')

In [34]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from qiime2 import Artifact
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from matplotlib.patches import Ellipse

# Define the color gradient for 'local_lesion_severity_mean_ID_CC' as a colormap
custom_colors = ["#423fa6", "#67b2be", "#ededed", "#df7963", "#ca0000", "#960000"]
severity_cmap = LinearSegmentedColormap.from_list("severity_gradient", custom_colors)

# Path to the pre-existing metadata file
mf_path = '/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Figures/16S_Figures/CTF/subject-metadata_subject_ID_mean_severity.tsv'

# Load the pre-existing metadata file
mf = pd.read_csv(mf_path, sep='\t', index_col='#SampleID')

# Function to draw ellipses
def draw_ellipse(mean, cov, ax, color, scale_factor=1):
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * scale_factor * np.sqrt(eigvals)
    ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                  edgecolor=color, facecolor=color, alpha=0.2, linewidth=0.5)
    ax.add_patch(ell)

# Run CTF and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Run CTF
    ctf_results = ctf(table, metadata, 'id_loc', 'day', feature_metadata=taxonomy)
    
    # Unpack results
    subject_biplot_artifact = ctf_results[0]
    distance_matrix_artifact = ctf_results[2]
    
    # Convert results to Pandas DataFrame
    distance_matrix = distance_matrix_artifact.view(DistanceMatrix)
    distance_df = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    subject_biplot = subject_biplot_artifact.view(OrdinationResults)
    
    # Extract coordinates
    spca_df = pd.DataFrame(subject_biplot.samples, index=subject_biplot.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Rename columns for clarity

    # Map 'local_lesion_severity_mean_ID_CC' from mf to spca_df
    spca_df["local_lesion_severity_mean_ID_CC"] = spca_df.index.map(mf["local_lesion_severity_mean_ID_CC"])

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Scatter plot with gradient color based on severity
    sc = ax.scatter(
        spca_df['PC1'],
        spca_df['PC2'],
        c=spca_df['local_lesion_severity_mean_ID_CC'],
        cmap=severity_cmap,
        edgecolor='k',
        linewidths=0.3,
        alpha=0.8,
        s=30
    )
    
    # Add color bar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Mean Local Lesion Severity per subject ID\'s C-zone', fontsize=7)
    cbar.ax.tick_params(labelsize=6)
    
    # Customize labels
    ax.set_xlabel(f'PC1 ({subject_biplot.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({subject_biplot.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    # Add title
    ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)
    
    # Final touches
    plt.tight_layout()
    save_path = os.path.join(output_path, f"{file_suffix}_local_lesion_severity_mean_ID_CC_continuous.png")
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure to free memory

    print(f"Metadata saved to {os.path.join(output_path, 'subject-metadata.tsv')}")
    print(f"Plot saved to {save_path}")


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()


Metadata saved to ../Figures/16S_Figures/CTF/v1_v3/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v1_v3/v1_v3_local_lesion_severity_mean_ID_CC_continuous.png


/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:425: RuntimeWarning: divide by zero encountered in log
  mat = np.log(matrix_closure(mat))
/Users/bpessemier/miniconda3/envs/qiime2-shotgun-2024.2_adapted/lib/python3.8/site-packages/gemelli/preprocessing.py:988: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  order_tmp = mf_tmp.groupby(grp_col).sum()


Metadata saved to ../Figures/16S_Figures/CTF/v4/subject-metadata.tsv
Plot saved to ../Figures/16S_Figures/CTF/v4/v4_local_lesion_severity_mean_ID_CC_continuous.png


In [31]:
from itertools import combinations


In [32]:
# Function to perform pairwise PERMANOVA
def perform_pairwise_permanova(distance, spca_df, groups_column):
    group_pairs = list(combinations(spca_df[groups_column].unique(), 2))
    permanova_results = {}
    
    for group1, group2 in group_pairs:
        # Subset the dataframe for the two groups
        subset = spca_df[spca_df[groups_column].isin([group1, group2])]
        # Filter the distance matrix for this subset
        distance_subset = distance.filter(subset.index)
        # Perform PERMANOVA on the filtered distance matrix for this subset
        result = permanova(distance_subset, subset, groups_column)
        permanova_results[f"{group1} vs {group2}"] = result['p-value']
    
    return permanova_results

# RPCA

In [14]:
# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_table_filt.filtered_table, q2_meta_v3, 'RPCA - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/RPCA/v1_v3'),
    (v4_table_filt.filtered_table, q2_meta_v4, 'RPCA - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/RPCA/v4')
]

# Update colors and group name mapping for new labels
group_colors = {
    "Healthy": "#423fa6",
    "Acne Lesional": "#df7963",
    "Acne Non-lesional": "#67b2be"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne Lesional": "Acne Lesional",
    "Acne Non-lesional": "Acne Non-lesional"
}

# Ensure output directories exist
for _, _, _, _, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)

In [16]:
# Function to extract last two parts of the taxon (if needed for plotting)
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]
    else:
        return ";".join(parts[-1:])


In [17]:
# Run RPCA and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Ensure output directory exists
    os.makedirs(output_path, exist_ok=True)
    
    # Run RPCA
    np.seterr(divide='ignore')
    ordination_artifact, distance = rpca(table)
    
    # Convert ordination artifact to OrdinationResults
    ordination = ordination_artifact.view(OrdinationResults)
    
    # Clear any existing plots
    plt.clf()

    # Extract sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Assign column names for plotting
    
    # Map group labels from metadata to spca_df using the index directly
    spca_df["Group"] = spca_df.index.map(metadata.to_dataframe()["category_label"])

    # Extract feature ordinations and assign column names
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Load taxonomy and extract short labels
    taxonomy = Artifact.load('../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    taxonomy["short_taxa"] = taxonomy["Taxon"].apply(extract_last_two_taxa)

    # Select top 10 features based on variance
    top_features = feature_coordinates.var(axis=1).sort_values(ascending=False).head(10)
    top_features_df = feature_coordinates.loc[top_features.index]
    top_features_with_taxa = top_features_df.merge(taxonomy[['short_taxa']], left_index=True, right_index=True)

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Scatter plot of sample ordinations with hue for Group
    sns.scatterplot(
        data=spca_df,
        x="PC1",
        y="PC2",
        hue="Group",
        palette=group_colors,
        hue_order=list(group_name_mapping.keys()),
        alpha=0.8,
        linewidth=0.5,
        ax=ax,
        s=30  # Adjusted dot size for clarity
    )

    # Draw ellipses and stars at the mean location for each group
    for group_name, group in spca_df.groupby("Group"):
        if group_name in group_colors:
            # Calculate mean and covariance matrix for the group
            points = group[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            cov = np.cov(points, rowvar=False)

            # Draw the ellipse for the group
            draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

            # Add a star at the mean location for each group
            ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Adjust arrow length for top 10 features
    arrow_scaling_factor = 0.3  # Reduce scaling factor to shorten arrows
    for feature_id in top_features_with_taxa.index:
        feature_coord = top_features_df.loc[feature_id, ["PC1", "PC2"]] * arrow_scaling_factor
        taxon_label = top_features_with_taxa.loc[feature_id, "short_taxa"]
        
        # Draw arrow and label for the feature
        ax.arrow(0, 0, feature_coord["PC1"], feature_coord["PC2"], color="grey", alpha=0.3, head_width=0.01, length_includes_head=True)
        ax.text(feature_coord["PC1"] * 1.1, feature_coord["PC2"] * 1.1, taxon_label, fontsize=5, color="black")

    # Customize plot labels and title
    ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    
    # Set tick parameters to reduce tick label size
    ax.tick_params(axis='both', which='major', labelsize=5)

    # Add title
    ax.text(0.05, 1.10, plot_title, fontsize=7, va='top', ha='left', transform=ax.transAxes)

    # Automatically adjust legend position and reduce font size
    plt.legend(
        frameon=False,
        fontsize=6,
        title_fontsize=6,
        loc='best',           # 'best' lets matplotlib choose the location automatically
        bbox_to_anchor=(1, 1) # Adjust as necessary
    )

    # Save the plot with "_rpca" suffix
    save_path = os.path.join(output_path, f"{file_suffix}_rpca.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Plot saved to {save_path}")


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_3535/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_3535/88303974.py:66: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_3535/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; t

Plot saved to ../Figures/16S_Figures/RPCA/v1_v3/v1_v3_rpca.png


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_3535/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_3535/88303974.py:66: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_3535/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; t

Plot saved to ../Figures/16S_Figures/RPCA/v4/v4_rpca.png


In [17]:
# Run RPCA and save metadata and plots for each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    # Ensure output directory exists
    os.makedirs(output_path, exist_ok=True)
    
    # Run RPCA
    np.seterr(divide='ignore')
    ordination_artifact, distance = rpca(table)
    
    # Convert ordination artifact to OrdinationResults
    ordination = ordination_artifact.view(OrdinationResults)
    
    # Clear any existing plots
    plt.clf()

    # Extract sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']  # Assign column names for plotting
    
    # Map group labels from metadata to spca_df using the index directly
    spca_df["Group"] = spca_df.index.map(metadata.to_dataframe()["category_label"])

    # Extract feature ordinations and assign column names
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Load taxonomy and extract short labels
    taxonomy = Artifact.load('../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    taxonomy["short_taxa"] = taxonomy["Taxon"].apply(extract_last_two_taxa)

    # Select top 10 features based on variance
    top_features = feature_coordinates.var(axis=1).sort_values(ascending=False).head(10)
    top_features_df = feature_coordinates.loc[top_features.index]
    top_features_with_taxa = top_features_df.merge(taxonomy[['short_taxa']], left_index=True, right_index=True)

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Scatter plot of sample ordinations with hue for Group
    sns.scatterplot(
        data=spca_df,
        x="PC1",
        y="PC2",
        hue="Group",
        palette=group_colors,
        hue_order=list(group_name_mapping.keys()),
        alpha=0.8,
        linewidth=0.5,
        ax=ax,
        s=30  # Adjusted dot size
    )

    # Draw ellipses and stars at the mean location for each group
    for group_name, group in spca_df.groupby("Group"):
        if group_name in group_colors:
            # Calculate mean and covariance matrix for the group
            points = group[["PC1", "PC2"]].values
            mean = np.mean(points, axis=0)
            cov = np.cov(points, rowvar=False)

            # Draw the ellipse for the group
            draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

            # Add a star at the mean location for each group
            ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Adjust arrow length for top 10 features
    arrow_scaling_factor = 0.2  # Reduce scaling factor to shorten arrows
    for feature_id in top_features_with_taxa.index:
        feature_coord = top_features_df.loc[feature_id, ["PC1", "PC2"]] * arrow_scaling_factor
        taxon_label = top_features_with_taxa.loc[feature_id, "short_taxa"]
        
        # Draw arrow and label for the feature
        ax.arrow(0, 0, feature_coord["PC1"], feature_coord["PC2"], color="grey", alpha=0.3, head_width=0.01, length_includes_head=True)
        ax.text(feature_coord["PC1"] * 1.1, feature_coord["PC2"] * 1.1, taxon_label, fontsize=5, color="black")

    # Customize plot labels and title
    ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    ax.tick_params(axis='both', which='major', labelsize=5)  # Reduce tick label size

    # Show only the x and y axes (remove top and right spines)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)

    # Add title
    ax.text(0.05, 1.10, plot_title, fontsize=7, va='top', ha='left', transform=ax.transAxes)

    # Automatically position legend in an empty area
    plt.legend(
        frameon=False,
        fontsize=6,
        title_fontsize=6,
        loc='best'
    )

    # Save the plot with "_rpca" suffix
    save_path = os.path.join(output_path, f"{file_suffix}_rpca.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close(fig)  # Close the figure after saving to free memory

    print(f"Plot saved to {save_path}")


NameError: name 'extract_last_two_taxa' is not defined

#including local_lesion_severity in the metadata

In [35]:
# first we import the metdata into pandas
mf = pd.read_csv('../data_Tyler/57610_57610_analysis_mapping.txt', sep='\t', index_col=0)
mf = mf.loc[mf.sample_type=='skin']
mf['local_lesion_severity']= mf[['visual_assessment_on_photography_acne_severity_cheek_left', 'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['acne_severity']= mf[['visual_assessment_on_photography_acne_severity_cheek_left', 'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['inflammatory_lesions_face']= mf[['visual_assessment_in_vivo_number_of_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['noninflammatory_lesions_face']= mf[['visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['a']= mf[['skincam_a_cheek_left','skincam_a_cheek_right']].replace('not applicable', 0).astype(float).sum(axis=1)

mf.loc[mf['subject_randomization_number'].astype(int)>299, 'cohort'] = 'acne'
mf.loc[mf['subject_randomization_number'].astype(int)<299, 'cohort'] = 'control'

mf['subject_randomization_id'] = mf['subject_randomization_number'].apply(lambda x: f"RD{int(x):03d}" if 0 <= int(x) <= 399 else x)

#mf=mf.drop(columns='Subject_Randomization_Number')
mf['class'] = mf['acne_severity'].apply(lambda x: 'acne' if x >= 1 else 'healthy')
mf['day'] = [int(day.replace('D', '')) if day != 'not applicable' else day for day in mf['day']]#mf['day'].str.replace('D', '').astype(int)

# Define the conditions for assigning values to the "category" column
condition1 = (mf['c_zone'] == 'C3')
condition2 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'acne'))
condition3 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'control'))

# Use the conditions to assign values to the "category" column
mf.loc[condition1, 'category'] = 'clear_zone'
mf.loc[condition2, 'category'] = 'acne'
mf.loc[condition3, 'category'] = 'control'

features_of_interest = ['a','skin_ph_meter_ph_mean_forehead_right', 'sebumeter_casual_level_mean_forehead_left',
                    'acne_severity', 'inflammatory_lesions_face', 'noninflammatory_lesions_face','day']#,'subject_randomization_id','subject_randomization_number', 'class', 'cohort', 'Current_Natural_Log_Ratio']

mf = mf.astype({col: 'str' for col in mf.columns[mf.dtypes == 'bool']})

In [18]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from qiime2 import Artifact, Metadata
from skbio.stats.ordination import OrdinationResults
from itertools import combinations
from matplotlib.patches import Ellipse

# Load and preprocess metadata
mf = pd.read_csv('../data_Tyler/57610_57610_analysis_mapping.txt', sep='\t', index_col=0)
mf = mf.loc[mf.sample_type == 'skin']

# Calculate severity-related columns
mf['local_lesion_severity'] = mf[['visual_assessment_on_photography_acne_severity_cheek_left', 
                                  'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['acne_severity'] = mf[['visual_assessment_on_photography_acne_severity_cheek_left', 
                          'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['inflammatory_lesions_face'] = mf[['visual_assessment_in_vivo_number_of_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['noninflammatory_lesions_face'] = mf[['visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['a'] = mf[['skincam_a_cheek_left', 'skincam_a_cheek_right']].replace('not applicable', 0).astype(float).sum(axis=1)

# Define cohorts and other columns
mf.loc[mf['subject_randomization_number'].astype(int) > 299, 'cohort'] = 'acne'
mf.loc[mf['subject_randomization_number'].astype(int) < 299, 'cohort'] = 'control'
mf['subject_randomization_id'] = mf['subject_randomization_number'].apply(lambda x: f"RD{int(x):03d}" if 0 <= int(x) <= 399 else x)
mf['class'] = mf['acne_severity'].apply(lambda x: 'acne' if x >= 1 else 'healthy')
mf['day'] = mf['day'].apply(lambda x: int(x.replace('D', '')) if x != 'not applicable' else x)

# Define conditions for "category" column
condition1 = (mf['c_zone'] == 'C3')
condition2 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'acne'))
condition3 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'control'))

# Assign values to "category" based on conditions
mf.loc[condition1, 'category'] = 'clear_zone'
mf.loc[condition2, 'category'] = 'acne'
mf.loc[condition3, 'category'] = 'control'

# Create a mapping for severity levels
severity_mapping = {
    0: 'absent',
    1: 'low',
    2: 'low',
    3: 'moderate',
    4: 'moderate',
    5: 'high',
    6: 'high'
}



# Ensure the boolean columns are converted to strings for Metadata compatibility
mf = mf.astype({col: 'str' for col in mf.columns[mf.dtypes == 'bool']})

# Create a new column 'category_label' for more descriptive labels
category_mapping = {
    'clear_zone': 'Acne Non-lesional',
    'acne': 'Acne Lesional',
    'control': 'Healthy'
}
mf['category_label'] = mf['category'].map(category_mapping)

# Apply severity mapping
mf['severity_level'] = mf['local_lesion_severity'].map(severity_mapping)
mf['severity_group'] = mf['severity_level'] + ' ' + mf['category_label']

# Use 'category_label' in metadata for the CTF analysis and plots
meta_v3 = mf.copy()
meta_v4 = mf.copy()

meta_v3['c_zone_group'] = meta_v3.cohort+'_'+meta_v3.c_zone
meta_v3['id_loc'] = meta_v3.subject_randomization_id+'_'+meta_v3.c_zone

meta_v4['c_zone_group'] = meta_v4.cohort+'_'+meta_v4.c_zone
meta_v4['id_loc'] = meta_v4.subject_randomization_id+'_'+meta_v4.c_zone

q2_meta_v3 = Metadata(meta_v3.applymap(lambda x: str(x) if isinstance(x, bool) else x))
q2_meta_v4 = Metadata(meta_v4.applymap(lambda x: str(x) if isinstance(x, bool) else x))

# Save the modified metadata as qiime2 Metadata for RPCA
metadata = Metadata.load('../data_Tyler/57610_57610_analysis_mapping.txt')

# Filter the tables
v3_table = Artifact.load('../data_Tyler/174950_rarefied_table.qza')
v4_table = Artifact.load('../data_Tyler/174951_rarefied_table.qza')
v3_table_filt = filter_samples(table=v3_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)
v4_table_filt = filter_samples(table=v4_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)

# Define colors and shapes for severity groups
severity_group_colors = {
    "absent Healthy": "#423fa6",
    "absent Acne Non-lesional": "#67b2be", 
    "low Acne Non-lesional": "#8ef0f1",
    "low Acne Lesional": "#df7963",
    "moderate Acne Lesional": "#ca0000", 
    "high Acne Lesional": "#960000"
}

group_shape_mapping = {
    "Healthy": "o",
    "Acne Non-lesional": "^",
    "Acne Lesional": "D"
}

severity_order = ["absent Healthy", "absent Acne Non-lesional", "low Acne Non-lesional", "low Acne Lesional", "moderate Acne Lesional", "high Acne Lesional"]

# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_table_filt.filtered_table, q2_meta_v3, 'RPCA - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/v1_v3'),
    (v4_table_filt.filtered_table, q2_meta_v4, 'RPCA - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/v4')
]

# Draw ellipse function for covariance-based ellipses around groups
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

# RPCA analysis and plotting
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)
    
    # Run RPCA
    np.seterr(divide='ignore')
    ordination_artifact, distance = rpca(table)
    ordination = ordination_artifact.view(OrdinationResults)
    plt.clf()

    # Extract sample ordinations and feature coordinates
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']
    spca_df["severity_group"] = spca_df.index.map(metadata.to_dataframe()["severity_group"])
    spca_df["category_label"] = spca_df.index.map(metadata.to_dataframe()["category_label"])

    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load taxonomy and prepare top 10 feature labels
    taxonomy = Artifact.load('../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    taxonomy["short_taxa"] = taxonomy["Taxon"].apply(lambda x: ";".join(x.split(";")[-2:]) if "uncultured" not in x else x.split(";")[-3])

    top_features = feature_coordinates.var(axis=1).sort_values(ascending=False).head(10)
    top_features_with_taxa = feature_coordinates.loc[top_features.index].merge(taxonomy[['short_taxa']], left_index=True, right_index=True)

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:
            group_data = spca_df[spca_df['severity_group'] == severity_group]
            for group_name, data in group_data.groupby('category_label'):
                sns.scatterplot(
                    data=data,
                    x="PC1",
                    y="PC2",
                    facecolor=severity_group_colors[severity_group],
                    edgecolor=severity_group_colors[severity_group],
                    alpha=0.8,
                    linewidth=0.5,
                    ax=ax,
                    label=severity_group,
                    marker=group_shape_mapping[group_name],
                    s=10
                )

    # Draw ellipses and feature vectors
    for group_name, data in spca_df.groupby("category_label"):
        points = data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        if len(data) > 2:
            cov = np.cov(points, rowvar=False)
            ellipse_color = {
                "Healthy": "#423fa6",
                "Acne Non-lesional": "#67b2be",   
                "Acne Lesional": "#df7963"     
            }[group_name]

            draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)
            
            #draw_ellipse(mean, cov, ax, severity_group_colors.get(group_name, "#999999"), scale_factor=2)

    for i, feature in top_features_with_taxa.iterrows():
        ax.arrow(0, 0, feature["PC1"] * 0.3, feature["PC2"] * 0.3, color="grey", head_width=0.01, length_includes_head=True, alpha=0.3)
        ax.text(feature["PC1"] * 0.33, feature["PC2"] * 0.33, feature["short_taxa"], fontsize=5, color="black")

    ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    ax.tick_params(axis='both', which='major', labelsize=5)


    # Remove top and right spines (the rectangular box)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.legend(
        frameon=False,
        fontsize=6,
        title_fontsize=6,
        loc='best'
    )

    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{file_suffix}_rpca_severity_group.png"), dpi=600)
    plt.close(fig)
    print(f"Plot saved to {output_path}")


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/1929905519.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/1929905519.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/1929905519.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will be

Plot saved to ../Figures/16S_Figures/CTF/v1_v3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/1929905519.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/1929905519.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/1929905519.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will be

Plot saved to ../Figures/16S_Figures/CTF/v4


In [19]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from qiime2 import Artifact, Metadata
from skbio.stats.ordination import OrdinationResults
from itertools import combinations
from matplotlib.patches import Ellipse

# Load and preprocess metadata
mf = pd.read_csv('../data_Tyler/57610_57610_analysis_mapping.txt', sep='\t', index_col=0)
mf = mf.loc[mf.sample_type == 'skin']

# Calculate severity-related columns
mf['local_lesion_severity'] = mf[['visual_assessment_on_photography_acne_severity_cheek_left', 
                                  'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['acne_severity'] = mf[['visual_assessment_on_photography_acne_severity_cheek_left', 
                          'visual_assessment_on_photography_acne_severity_cheek_right']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['inflammatory_lesions_face'] = mf[['visual_assessment_in_vivo_number_of_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['noninflammatory_lesions_face'] = mf[['visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face']].replace('not applicable', 0).astype(int).sum(axis=1)
mf['a'] = mf[['skincam_a_cheek_left', 'skincam_a_cheek_right']].replace('not applicable', 0).astype(float).sum(axis=1)

# Define cohorts and other columns
mf.loc[mf['subject_randomization_number'].astype(int) > 299, 'cohort'] = 'acne'
mf.loc[mf['subject_randomization_number'].astype(int) < 299, 'cohort'] = 'control'
mf['subject_randomization_id'] = mf['subject_randomization_number'].apply(lambda x: f"RD{int(x):03d}" if 0 <= int(x) <= 399 else x)
mf['class'] = mf['acne_severity'].apply(lambda x: 'acne' if x >= 1 else 'healthy')
mf['day'] = mf['day'].apply(lambda x: int(x.replace('D', '')) if x != 'not applicable' else x)

# Define conditions for "category" column
condition1 = (mf['c_zone'] == 'C3')
condition2 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'acne'))
condition3 = (mf['c_zone'].isin(['C1', 'C2']) & (mf['cohort'] == 'control'))

# Assign values to "category" based on conditions
mf.loc[condition1, 'category'] = 'clear_zone'
mf.loc[condition2, 'category'] = 'acne'
mf.loc[condition3, 'category'] = 'control'

# Create a mapping for severity levels
severity_mapping = {
    0: 'absent',
    1: 'low',
    2: 'low',
    3: 'moderate',
    4: 'moderate',
    5: 'high',
    6: 'high'
}



# Ensure the boolean columns are converted to strings for Metadata compatibility
mf = mf.astype({col: 'str' for col in mf.columns[mf.dtypes == 'bool']})

# Create a new column 'category_label' for more descriptive labels
category_mapping = {
    'clear_zone': 'Acne Non-lesional',
    'acne': 'Acne Lesional',
    'control': 'Healthy'
}
mf['category_label'] = mf['category'].map(category_mapping)

# Apply severity mapping
mf['severity_level'] = mf['local_lesion_severity'].map(severity_mapping)
mf['severity_group'] = mf['severity_level'] + ' ' + mf['category_label']

# Use 'category_label' in metadata for the CTF analysis and plots
meta_v3 = mf.copy()
meta_v4 = mf.copy()

meta_v3['c_zone_group'] = meta_v3.cohort+'_'+meta_v3.c_zone
meta_v3['id_loc'] = meta_v3.subject_randomization_id+'_'+meta_v3.c_zone

meta_v4['c_zone_group'] = meta_v4.cohort+'_'+meta_v4.c_zone
meta_v4['id_loc'] = meta_v4.subject_randomization_id+'_'+meta_v4.c_zone

q2_meta_v3 = Metadata(meta_v3.applymap(lambda x: str(x) if isinstance(x, bool) else x))
q2_meta_v4 = Metadata(meta_v4.applymap(lambda x: str(x) if isinstance(x, bool) else x))

# Save the modified metadata as qiime2 Metadata for RPCA
metadata = Metadata.load('../data_Tyler/57610_57610_analysis_mapping.txt')

# Filter the tables
v3_table = Artifact.load('../data_Tyler/174950_rarefied_table.qza')
v4_table = Artifact.load('../data_Tyler/174951_rarefied_table.qza')
v3_table_filt = filter_samples(table=v3_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)
v4_table_filt = filter_samples(table=v4_table, metadata=metadata, where='sample_type!="skin"', exclude_ids=True)

# Define colors and shapes for severity groups
severity_group_colors = {
    "absent Healthy": "#423fa6",
    "absent Acne Non-lesional": "#67b2be", 
    "low Acne Non-lesional": "#8ef0f1",
    "low Acne Lesional": "#df7963",
    "moderate Acne Lesional": "#ca0000", 
    "high Acne Lesional": "#960000"
}

group_shape_mapping = {
    "Healthy": "o",
    "Acne Non-lesional": "^",
    "Acne Lesional": "D"
}

severity_order = ["absent Healthy", "absent Acne Non-lesional", "low Acne Non-lesional", "low Acne Lesional", "moderate Acne Lesional", "high Acne Lesional"]

# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_table_filt.filtered_table, q2_meta_v3, 'RPCA - 16S rRNA (V1-V3)', 'v1_v3', '../Figures/16S_Figures/CTF/v1_v3'),
    (v4_table_filt.filtered_table, q2_meta_v4, 'RPCA - 16S rRNA (V4)', 'v4', '../Figures/16S_Figures/CTF/v4')
]

# Draw ellipse function for covariance-based ellipses around groups
def draw_ellipse(mean, cov, ax, color, label=None, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor
    ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
    ax.add_patch(ell)

# RPCA analysis and plotting
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)
    
    # Run RPCA
    np.seterr(divide='ignore')
    ordination_artifact, distance = rpca(table)
    ordination = ordination_artifact.view(OrdinationResults)
    plt.clf()

    # Extract sample ordinations and feature coordinates
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df.columns = ['PC1', 'PC2', 'PC3']
    spca_df["severity_group"] = spca_df.index.map(metadata.to_dataframe()["severity_group"])
    spca_df["category_label"] = spca_df.index.map(metadata.to_dataframe()["category_label"])

    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

    # Load taxonomy and prepare top 10 feature labels
    taxonomy = Artifact.load('../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    taxonomy["short_taxa"] = taxonomy["Taxon"].apply(lambda x: ";".join(x.split(";")[-2:]) if "uncultured" not in x else x.split(";")[-3])

    top_features = feature_coordinates.var(axis=1).sort_values(ascending=False).head(10)
    top_features_with_taxa = feature_coordinates.loc[top_features.index].merge(taxonomy[['short_taxa']], left_index=True, right_index=True)

    # Plotting
    mm = 1 / 25.4
    fig, ax = plt.subplots(1, 1, figsize=(90 * mm, 110 * mm))
    
    # Scatter plot for each severity group with its specific color
    for severity_group in severity_order:
        if severity_group in spca_df['severity_group'].values:
            group_data = spca_df[spca_df['severity_group'] == severity_group]
            for group_name, data in group_data.groupby('category_label'):
                sns.scatterplot(
                    data=data,
                    x="PC1",
                    y="PC2",
                    facecolor=severity_group_colors[severity_group],
                    edgecolor=severity_group_colors[severity_group],
                    alpha=0.8,
                    linewidth=0.5,
                    ax=ax,
                    label=severity_group,
                    marker=group_shape_mapping[group_name],
                    s=10
                )

    # Draw ellipses based on category_label and add stars at centers for severity groups
    # Ellipses by category_label
    for category_label, data in spca_df.groupby("category_label"):
        points = data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)

        if len(data) > 2:
            cov = np.cov(points, rowvar=False)
            # Define the color of the ellipse based on category_label
            ellipse_color = {
                "Healthy": "#423fa6",
                "Acne Non-lesional": "#67b2be",   
                "Acne Lesional": "#df7963"     
            }[category_label]

            # Draw the ellipse for each category_label
            draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)

    # Add stars based on severity_group
    for severity_group, data in spca_df.groupby("severity_group"):
        points = data[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        
        # Use the color defined for each severity_group
        star_color = severity_group_colors.get(severity_group, "#999999")
        
        # Add a star at the mean location for each severity_group
        ax.scatter(mean[0], mean[1], color=star_color, marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

    # Plot feature vectors
    #for i, feature in top_features_with_taxa.iterrows():
    #    ax.arrow(0, 0, feature["PC1"] * 0.3, feature["PC2"] * 0.3, color="grey", head_width=0.01, length_includes_head=True, alpha=0.3)
    #    ax.text(feature["PC1"] * 0.33, feature["PC2"] * 0.33, feature["short_taxa"], fontsize=5, color="black")

    ax.set_xlabel(f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)', fontsize=7)
    ax.set_ylabel(f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)', fontsize=7)
    ax.tick_params(axis='both', which='major', labelsize=5)

    # Remove top and right spines (the rectangular box)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.legend(
        frameon=False,
        fontsize=6,
        title_fontsize=6,
        loc='best'#,
        #bbox_to_anchor=(1, 1)
    )

    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f"{file_suffix}_rpca_severity_group.png"), dpi=600)
    plt.close(fig)
    print(f"Plot saved to {output_path}")


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/3557189431.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/3557189431.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/3557189431.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will be

Plot saved to ../Figures/16S_Figures/CTF/v1_v3


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/3557189431.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/3557189431.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_19939/3557189431.py:123: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will be

Plot saved to ../Figures/16S_Figures/CTF/v4


In [20]:
from skbio.stats.distance import permanova, DistanceMatrix

# Define pairwise PERMANOVA function
def perform_pairwise_permanova(distance_matrix, spca_df, groups_column, alpha=0.05):
    group_pairs = list(combinations(spca_df[groups_column].unique(), 2))
    permanova_results = {}
    
    for group1, group2 in group_pairs:
        subset = spca_df[spca_df[groups_column].isin([group1, group2])]
        distance_subset = distance_matrix.filter(subset.index)
        result = permanova(distance_subset, subset, groups_column)
        
        p_value = result.get('p-value', None)
        f_measure = result.get('test statistic', None)
        
        if p_value is not None and f_measure is not None and p_value < alpha:
            permanova_results[f"{group1} vs {group2}"] = (p_value, f_measure)
        elif p_value is not None and p_value < alpha:
            permanova_results[f"{group1} vs {group2}"] = (p_value, None)
    
    return permanova_results

# Loop over each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)
    
    # Run RPCA
    np.seterr(divide='ignore')
    ordination_artifact, distance_artifact = rpca(table)
    ordination = ordination_artifact.view(OrdinationResults)
    distance_matrix = distance_artifact.view(DistanceMatrix)  # Convert to DistanceMatrix
    
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df["severity_group"] = spca_df.index.map(metadata.to_dataframe()["severity_group"])
    
    # Perform pairwise PERMANOVA for severity groups
    pairwise_permanova_results = perform_pairwise_permanova(distance_matrix, spca_df, "severity_group")
    
    # Display pairwise PERMANOVA results
    print(f"Pairwise PERMANOVA results for {plot_title}:")
    for comparison, (p_value, f_measure) in pairwise_permanova_results.items():
        print(f"{comparison}: p-value = {p_value:.3f}, F-measure = {f_measure if f_measure is not None else 'N/A'}")


Pairwise PERMANOVA results for RPCA - 16S rRNA (V1-V3):
absent Healthy vs moderate Acne Lesional: p-value = 0.007, F-measure = 5.60882817064371
absent Healthy vs absent Acne Non-lesional: p-value = 0.006, F-measure = 6.6970272960374455
absent Healthy vs low Acne Lesional: p-value = 0.002, F-measure = 8.297332061128865
absent Healthy vs low Acne Non-lesional: p-value = 0.001, F-measure = 10.236227918287797
absent Healthy vs high Acne Lesional: p-value = 0.004, F-measure = 6.245232069115385
low Acne Lesional vs low Acne Non-lesional: p-value = 0.041, F-measure = 3.038105786989266
Pairwise PERMANOVA results for RPCA - 16S rRNA (V4):
absent Healthy vs moderate Acne Lesional: p-value = 0.022, F-measure = 3.8893701912974485
absent Healthy vs low Acne Lesional: p-value = 0.002, F-measure = 11.39543741568179
absent Healthy vs absent Acne Non-lesional: p-value = 0.012, F-measure = 4.974298668159867
absent Healthy vs high Acne Lesional: p-value = 0.006, F-measure = 5.658707076957919
absent Healt

In [21]:
from skbio.stats.distance import permanova, DistanceMatrix

# Define pairwise PERMANOVA function to include all results
def perform_pairwise_permanova(distance_matrix, spca_df, groups_column):
    group_pairs = list(combinations(spca_df[groups_column].unique(), 2))
    permanova_results = {}
    
    for group1, group2 in group_pairs:
        subset = spca_df[spca_df[groups_column].isin([group1, group2])]
        distance_subset = distance_matrix.filter(subset.index)
        result = permanova(distance_subset, subset, groups_column)
        
        p_value = result.get('p-value', None)
        f_measure = result.get('test statistic', None)
        
        # Store all results, including non-significant ones
        permanova_results[f"{group1} vs {group2}"] = (p_value, f_measure)
    
    return permanova_results

# Loop over each dataset
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    os.makedirs(output_path, exist_ok=True)
    
    # Run RPCA
    np.seterr(divide='ignore')
    ordination_artifact, distance_artifact = rpca(table)
    ordination = ordination_artifact.view(OrdinationResults)
    distance_matrix = distance_artifact.view(DistanceMatrix)  # Convert to DistanceMatrix
    
    spca_df = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    spca_df["severity_group"] = spca_df.index.map(metadata.to_dataframe()["severity_group"])
    
    # Perform pairwise PERMANOVA for severity groups
    pairwise_permanova_results = perform_pairwise_permanova(distance_matrix, spca_df, "severity_group")
    
    # Display pairwise PERMANOVA results, including non-significant ones
    print(f"Pairwise PERMANOVA results for {plot_title}:")
    for comparison, (p_value, f_measure) in pairwise_permanova_results.items():
        significance = " (significant)" if p_value < 0.05 else " (not significant)"
        print(f"{comparison}: p-value = {p_value:.3f}, F-measure = {f_measure if f_measure is not None else 'N/A'}{significance}")


Pairwise PERMANOVA results for RPCA - 16S rRNA (V1-V3):
absent Healthy vs moderate Acne Lesional: p-value = 0.006, F-measure = 5.608828170643718 (significant)
absent Healthy vs absent Acne Non-lesional: p-value = 0.008, F-measure = 6.697027296037438 (significant)
absent Healthy vs low Acne Lesional: p-value = 0.001, F-measure = 8.297332061128877 (significant)
absent Healthy vs low Acne Non-lesional: p-value = 0.001, F-measure = 10.23622791828779 (significant)
absent Healthy vs high Acne Lesional: p-value = 0.004, F-measure = 6.245232069115363 (significant)
moderate Acne Lesional vs absent Acne Non-lesional: p-value = 0.668, F-measure = 0.39358428946363305 (not significant)
moderate Acne Lesional vs low Acne Lesional: p-value = 0.543, F-measure = 0.5862082863078543 (not significant)
moderate Acne Lesional vs low Acne Non-lesional: p-value = 0.144, F-measure = 1.9878910592973182 (not significant)
moderate Acne Lesional vs high Acne Lesional: p-value = 0.480, F-measure = 0.731917689834586

In [45]:
print("Unique severity_group values in spca_df:", spca_df['severity_group'].unique())
print("Expected severity_group values (severity_order):", severity_order)

# Correct any differences if needed, otherwise proceed with plotting
missing_colors = [group for group in spca_df['severity_group'].unique() if group not in severity_group_colors]
if missing_colors:
    print("Warning: Some severity_group values are missing color mappings:", missing_colors)


Unique severity_group values in spca_df: ['absent Healthy' 'moderate Acne Lesional' 'low Acne Lesional'
 'absent Acne Non-lesional' 'high Acne Lesional' 'low Acne Non-lesional']
Expected severity_group values (severity_order): ['absent Healthy', 'absent Acne_NL', 'low Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L']


# CTF analysis Metabolomics with features

In [ ]:
# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    # Convert the biom table's data to a dense array and calculate feature prevalence
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]  # Convert to 1D array using .A1
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))

In [160]:

# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",
    "Acne_NL": "#67b2be",   # Adjusted name for Non-lesional
    "Acne_L": "#df7963"     # Adjusted name for Lesional
}


# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}


# Load the distance matrix artifact
table = Artifact.load(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/distance_matrix.qza')
    
# View as a skbio DistanceMatrix object
distance_matrix = table.view(DistanceMatrix)
    
# Convert the skbio DistanceMatrix to a Pandas DataFrame
data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
# Display the DataFrame
print(data)
    
# Load the subject_biplot artifact
biplot = Artifact.load(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/subject_biplot.qza')
    
# View as an OrdinationResults object
ordination = biplot.view(OrdinationResults)
    
# Extract the sample coordinates as a DataFrame
sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
# Extract the feature (or variable) coordinates as a DataFrame
feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
# Display the DataFrames
print("Sample Coordinates:")
print(sample_coordinates)
print("\nFeature Coordinates:")
print(feature_coordinates)
    
# Extract the proportion explained
proportion_explained = ordination.proportion_explained
    
# Convert to percentages for PC1 and PC2
pc1_explained_variance = proportion_explained[0] * 100
pc2_explained_variance = proportion_explained[1] * 100
    
print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
# Rename the sample coordinates DataFrame to spca_df
spca_df = sample_coordinates
    
# Load the metadata from the subject-metadata.tsv file
metadata_df = pd.read_csv(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/subject-metadata.tsv', sep='\t')
    
# Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
# Rename the columns of spca_df for clarity (PC1, PC2, PC3)
spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
# Display the updated spca_df
print(spca_df)

# Load and filter the BIOM table by prevalence
biom_path = f'/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Output/data_sample_method2_T.biom'
biom_table = load_table(biom_path)
prevalence = calculate_prevalence(biom_table)
prevalent_features = prevalence[prevalence >= 0.1].index  # Minimum 10% prevalence
    
# Filter feature coordinates by prevalent features
feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
feature_coordinates = feature_coordinates.loc[feature_coordinates.index.isin(prevalent_features)]
feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    

# Load taxonomy data
taxonomy = pd.read_csv("../Data/metabolomics/Run3_10252024/info_feature_complete.csv")

# Map Compound_Name and mz to the feature_coordinates DataFrame
feature_coordinates['Compound_Name'] = feature_coordinates.index.map(
    taxonomy.set_index('Feature')['Compound_Name']
)
feature_coordinates['mz'] = feature_coordinates.index.map(
    taxonomy.set_index('Feature')['mz'].apply(lambda x: f"mz: {x:.3f}" if not pd.isna(x) else None)
)

# Select top 10 features by magnitude for plotting
top_features = feature_coordinates[['PC1', 'PC2']].apply(
    lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1
).nlargest(10).index
top_feature_coords = feature_coordinates.loc[top_features]

# Plotting
mm = 1/25.4
fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
# Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    sns.scatterplot(
        data=group,
        x="PC1",
        y="PC2",
        facecolor=group_colors[group_name],  # Semi-transparent fill color
        edgecolor=group_colors[group_name],  # Colored border
        alpha=0.8,  # Transparency for the fill
        linewidth=0.5,  # Thickness of the borders
        ax=ax,
        label=group_name_mapping[group_name]
    )
    
    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)
    
    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    

    # Plot feature arrows for top prevalent features
    for feature in top_feature_coords.index:
        ax.arrow(
            0, 0, 
            top_feature_coords.loc[feature, 'PC1'], 
            top_feature_coords.loc[feature, 'PC2'],
            color='grey', alpha=0.3, head_width=0.02, length_includes_head=True
        )
        # Combine Compound_Name and mz for annotation
        compound_name = top_feature_coords.loc[feature, 'Compound_Name']
        mz = top_feature_coords.loc[feature, 'mz']
        annotation = f"{compound_name}\n{mz}" if pd.notna(compound_name) else mz
        ax.text(
            top_feature_coords.loc[feature, 'PC1'] * 1.1,
            top_feature_coords.loc[feature, 'PC2'] * 1.1,
            annotation,
            fontsize=3, color='black', ha="center"
        )
    
        

        
        
# Add axis labels and include the explained variance percentages
ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
# Simplify the ticks to avoid overcrowding
yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_yticks(yticklabels)
ax.set_yticklabels(yticklabels, fontsize=7)
    
xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_xticks(xticklabels)
ax.set_xticklabels(xticklabels, fontsize=7)
    
# Define the title
plot_title = "CTF Analysis Metabolomics"

# Add the title consistently in the top-left corner using axes coordinates
ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

# Customize legend
plt.legend(
    frameon=False,
    fontsize=7,
    title_fontsize=7
)
    
# Adjust plot style and save
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
    
plt.tight_layout()
plt.savefig(f"../Figures/metabolomics_Figures/CTF_method2.png", dpi=600)
plt.show()


                   LAMI.RD004.D0.C1  LAMI.RD001.D14.C2  LAMI.RD002.D0.C1  \
LAMI.RD004.D0.C1           0.000000          13.979833          8.442741   
LAMI.RD001.D14.C2         13.979833           0.000000          9.323204   
LAMI.RD002.D0.C1           8.442741           9.323204          0.000000   
LAMI.RD002.D14.C1          8.923500           6.757655          2.654330   
LAMI.RD011.D14.C1         13.347868          26.760449         19.083720   
...                             ...                ...               ...   
LAMI.RD318.D11.C1          6.851364          10.775982         10.703937   
LAMI.RD318.D16.C1          6.125530          11.797851         10.949374   
LAMI.RD318.D0.C1           5.778915          11.322872         10.265710   
LAMI.RD318.D14.C1          8.535246          13.086583         13.493557   
LAMI.RD318.D28.C1          7.242785          15.987839         14.333489   

                   LAMI.RD002.D14.C1  LAMI.RD011.D14.C1  LAMI.RD011.D28.C1  \
LAMI.RD00

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/678945592.py:104: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/678945592.py:129: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker

In [169]:
from adjustText import adjust_text  # Install with `pip install adjustText`


In [186]:
# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",
    "Acne_NL": "#67b2be",   # Adjusted name for Non-lesional
    "Acne_L": "#df7963"     # Adjusted name for Lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# Load the distance matrix artifact
table = Artifact.load(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/distance_matrix.qza')

# View as a skbio DistanceMatrix object
distance_matrix = table.view(DistanceMatrix)

# Convert the skbio DistanceMatrix to a Pandas DataFrame
data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)

# Display the DataFrame
print(data)

# Load the subject_biplot artifact
biplot = Artifact.load(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/subject_biplot.qza')

# View as an OrdinationResults object
ordination = biplot.view(OrdinationResults)

# Extract the sample coordinates as a DataFrame
sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)

# Extract the feature (or variable) coordinates as a DataFrame
feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)

# Display the DataFrames
print("Sample Coordinates:")
print(sample_coordinates)
print("\nFeature Coordinates:")
print(feature_coordinates)

# Extract the proportion explained
proportion_explained = ordination.proportion_explained

# Convert to percentages for PC1 and PC2
pc1_explained_variance = proportion_explained[0] * 100
pc2_explained_variance = proportion_explained[1] * 100

print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")

# Rename the sample coordinates DataFrame to spca_df
spca_df = sample_coordinates

# Load the metadata from the subject-metadata.tsv file
metadata_df = pd.read_csv(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/subject-metadata.tsv', sep='\t')

# Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])

# Rename the columns of spca_df for clarity (PC1, PC2, PC3)
spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']

# Display the updated spca_df
print(spca_df)

# Load and filter the BIOM table by prevalence
biom_path = f'/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Output/data_sample_method2_T.biom'
biom_table = load_table(biom_path)
prevalence = calculate_prevalence(biom_table)
prevalent_features = prevalence[prevalence >= 0].index  # Minimum 10% prevalence

# Filter feature coordinates by prevalent features
feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
feature_coordinates = feature_coordinates.loc[feature_coordinates.index.isin(prevalent_features)]
feature_coordinates.columns = ['PC1', 'PC2', 'PC3']

# Load taxonomy data
taxonomy = pd.read_csv("../Data/metabolomics/Run3_10252024/info_feature_complete.csv")

# Standardize taxonomy 'Feature' column to string for consistent mapping
taxonomy['Feature'] = taxonomy['Feature'].astype(str)

# Debug: Check if taxonomy features are now strings
print("Sample of taxonomy features after conversion:", taxonomy['Feature'].tolist()[:5])

# Set taxonomy index to 'Feature' for mapping
taxonomy = taxonomy.set_index('Feature')

# Map Compound_Name and mz to the feature_coordinates DataFrame
feature_coordinates['Compound_Name'] = feature_coordinates.index.map(
    taxonomy['Compound_Name'] if 'Compound_Name' in taxonomy.columns else None
)
feature_coordinates['mz'] = feature_coordinates.index.map(
    taxonomy['mz'] if 'mz' in taxonomy.columns else None
)

# Debug: Check mapped values
print("Sample mapped values in feature_coordinates:")
print(feature_coordinates[['Compound_Name', 'mz']].head())

# Filter features to exclude rows where both Compound_Name and mz are NaN
filtered_features = feature_coordinates[
    feature_coordinates['Compound_Name'].notna() | feature_coordinates['mz'].notna()
]

# Debug: Check number of features after filtering
print(f"Number of features after filtering: {len(filtered_features)}")

# Select top 10 features by magnitude for plotting
filtered_features['Magnitude'] = filtered_features[['PC1', 'PC2']].apply(
    lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1
)
top_features = filtered_features.nlargest(20, 'Magnitude')

# Debug: Check the top features
print("Top features for arrows and annotations:")
print(top_features[['PC1', 'PC2', 'Compound_Name', 'mz']])

# Plotting
mm = 1/25.4
fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))

# Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    sns.scatterplot(
        data=group,
        x="PC1",
        y="PC2",
        facecolor=group_colors[group_name],  # Semi-transparent fill color
        edgecolor=group_colors[group_name],  # Colored border
        alpha=0.8,  # Transparency for the fill
        linewidth=0.5,  # Thickness of the borders
        ax=ax,
        label=group_name_mapping[group_name]
    )

    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)

    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)

    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)

# Initialize a list to store annotation objects for overlap adjustment
annotations = []

# Plot feature arrows for top prevalent features
for feature in top_features.index:
    # Draw arrows with reduced thickness
    ax.arrow(
        0, 0,
        top_features.loc[feature, 'PC1'],
        top_features.loc[feature, 'PC2'],
        color='grey', alpha=0.5, linewidth=0.5, head_width=0.01, length_includes_head=True
    )

    # Combine Compound_Name, FeatureID, and mz for annotation
    compound_name = top_features.loc[feature, 'Compound_Name']
    mz = f"mz: {float(top_features.loc[feature, 'mz']):.3f}" if pd.notna(top_features.loc[feature, 'mz']) else ""
    annotation = f"{compound_name}\n{mz}" if pd.notna(compound_name) else f"{feature}\n{mz}"

    # Add the annotation text at the arrow tip
    text = ax.text(
        top_features.loc[feature, 'PC1'] * 1.05,
        top_features.loc[feature, 'PC2'] * 1.05,
        annotation,
        fontsize=3, color='black', ha="center"
    )
    annotations.append(text)

# Adjust annotations to avoid overlap
adjust_text(annotations, arrowprops=dict(arrowstyle="->", color='grey', lw=0.5))

# Add axis labels and include the explained variance percentages
ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)

# Simplify the ticks to avoid overcrowding
yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_yticks(yticklabels)
ax.set_yticklabels(yticklabels, fontsize=7)

xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_xticks(xticklabels)
ax.set_xticklabels(xticklabels, fontsize=7)

# Define the title
plot_title = "CTF Analysis Metabolomics"

# Add the title consistently in the top-left corner using axes coordinates
ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

# Customize legend
plt.legend(
    frameon=False,
    fontsize=7,
    title_fontsize=7
)

# Adjust plot style and save
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

plt.tight_layout()
plt.savefig(f"../Figures/metabolomics_Figures/CTF_method2.png", dpi=600)
plt.show()

                   LAMI.RD004.D0.C1  LAMI.RD001.D14.C2  LAMI.RD002.D0.C1  \
LAMI.RD004.D0.C1           0.000000          13.979833          8.442741   
LAMI.RD001.D14.C2         13.979833           0.000000          9.323204   
LAMI.RD002.D0.C1           8.442741           9.323204          0.000000   
LAMI.RD002.D14.C1          8.923500           6.757655          2.654330   
LAMI.RD011.D14.C1         13.347868          26.760449         19.083720   
...                             ...                ...               ...   
LAMI.RD318.D11.C1          6.851364          10.775982         10.703937   
LAMI.RD318.D16.C1          6.125530          11.797851         10.949374   
LAMI.RD318.D0.C1           5.778915          11.322872         10.265710   
LAMI.RD318.D14.C1          8.535246          13.086583         13.493557   
LAMI.RD318.D28.C1          7.242785          15.987839         14.333489   

                   LAMI.RD002.D14.C1  LAMI.RD011.D14.C1  LAMI.RD011.D28.C1  \
LAMI.RD00

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 3.6; the parameter will become keyword-only two minor releases later.
  ell = Ellipse(mean, width, height, angle, edgecolor=color, facecolor=color, alpha=0.2, label=label, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2768280265.py:150: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ((8, 2, 0)).  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/3313574044.py:15: MatplotlibDeprecationWarning: Passing the angle parameter of __init__() positionally is deprecated since Matplotlib 

In [165]:
# Debug: Check if taxonomy and feature_coordinates align
taxonomy_features = taxonomy['Feature'].tolist()
ordination_features = feature_coordinates.index.tolist()

# Print a sample to verify alignment
print("Sample of taxonomy features:", taxonomy_features[:5])
print("Sample of ordination features:", ordination_features[:5])

# Check for mismatches
mismatched_features = [feat for feat in ordination_features if feat not in taxonomy_features]
print("Mismatched features (if any):", mismatched_features)


Sample of taxonomy features: [39, 4031, 14841, 32751, 29425]
Sample of ordination features: ['262', '1074', '7111', '20210', '3496']
Mismatched features (if any): ['262', '1074', '7111', '20210', '3496', '23230', '12591', '27029', '1119', '26441', '954', '18532', '1368', '23740', '26201', '9352', '5907', '24042', '25915', '12132', '1661', '26788', '5856', '2969', '26277', '23625', '24992', '26782', '25893', '22958', '10878', '27183', '5002', '3513', '25280', '1178', '108', '12595', '21641', '1363', '3246', '15778', '26338', '5930', '25733', '26684', '7951', '10694', '29059', '21794', '24207', '25946', '26585', '5873', '5931', '25914', '9619', '19386', '26757', '2167', '23549', '26314', '675', '11885', '2965', '26565', '1470', '13332', '25374', '26659', '29440', '3401', '25822', '1060', '19801', '21023', '11856', '26605', '31119', '20255', '30538', '261', '19753', '14695', '862', '25423', '30218', '21047', '12806', '26295', '2153', '24584', '15063', '1267', '19727', '9912', '11462', '12

In [166]:
# Convert taxonomy Feature column to string for consistent mapping
taxonomy['Feature'] = taxonomy['Feature'].astype(str)

# Debug: Check if taxonomy features are now strings
print("Sample of taxonomy features after conversion:", taxonomy['Feature'].tolist()[:5])

# Map Compound_Name and mz to the feature_coordinates DataFrame
feature_coordinates['Compound_Name'] = feature_coordinates.index.map(
    taxonomy.set_index('Feature')['Compound_Name']
)
feature_coordinates['mz'] = feature_coordinates.index.map(
    taxonomy.set_index('Feature')['mz']
)

# Debug: Check if mapping was successful
print("Sample mapped values in feature_coordinates:")
print(feature_coordinates[['Compound_Name', 'mz']].head())

# Filter out features where both Compound_Name and mz are NaN
filtered_features = feature_coordinates[
    feature_coordinates['Compound_Name'].notna() | feature_coordinates['mz'].notna()
]

# Debug: Check number of features after filtering
print(f"Number of features after filtering: {len(filtered_features)}")

# Select top features by magnitude for plotting
filtered_features['Magnitude'] = filtered_features[['PC1', 'PC2']].apply(
    lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1
)
top_features = filtered_features.nlargest(10, 'Magnitude')

# Debug: Check the coordinates of the top features
print("Top features for arrows and annotations:")
print(top_features[['PC1', 'PC2', 'Compound_Name', 'mz']])


Sample of taxonomy features after conversion: ['39', '4031', '14841', '32751', '29425']
Sample mapped values in feature_coordinates:
      Compound_Name          mz
262             NaN  139.050010
1074            NaN  121.039477
7111            NaN  130.158855
20210           NaN  353.099149
3496   DL-Panthenol  206.138447
Number of features after filtering: 1137
Top features for arrows and annotations:
            PC1       PC2 Compound_Name          mz
262   -0.026153 -0.171778           NaN  139.050010
30644 -0.109533  0.067883           NaN  491.372880
26713  0.047498 -0.116424           NaN  636.397571
2531  -0.097086  0.076698           NaN  114.054819
1063  -0.107560  0.059238           NaN  188.127856
28042  0.059702 -0.106254           NaN  510.399738
6306   0.069413 -0.100113           NaN  370.170398
19926  0.069580 -0.094492           NaN  459.487665
26314 -0.111853  0.034686           NaN  480.388935
28290  0.049608 -0.105586           NaN  672.524869


# CTF Metabolomics without Features 

In [ ]:

# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",
    "Acne_NL": "#67b2be",   # Adjusted name for Non-lesional
    "Acne_L": "#df7963"     # Adjusted name for Lesional
}


# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}


# Load the distance matrix artifact
table = Artifact.load(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/distance_matrix.qza')
    
# View as a skbio DistanceMatrix object
distance_matrix = table.view(DistanceMatrix)
    
# Convert the skbio DistanceMatrix to a Pandas DataFrame
data = pd.DataFrame(distance_matrix.data, index=distance_matrix.ids, columns=distance_matrix.ids)
    
# Display the DataFrame
print(data)
    
# Load the subject_biplot artifact
biplot = Artifact.load(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/subject_biplot.qza')
    
# View as an OrdinationResults object
ordination = biplot.view(OrdinationResults)
    
# Extract the sample coordinates as a DataFrame
sample_coordinates = pd.DataFrame(ordination.samples, index=ordination.samples.index)
    
# Extract the feature (or variable) coordinates as a DataFrame
feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    
# Display the DataFrames
print("Sample Coordinates:")
print(sample_coordinates)
print("\nFeature Coordinates:")
print(feature_coordinates)
    
# Extract the proportion explained
proportion_explained = ordination.proportion_explained
    
# Convert to percentages for PC1 and PC2
pc1_explained_variance = proportion_explained[0] * 100
pc2_explained_variance = proportion_explained[1] * 100
    
print(f"PC1 explains {pc1_explained_variance:.2f}% of the variance")
print(f"PC2 explains {pc2_explained_variance:.2f}% of the variance")
    
# Rename the sample coordinates DataFrame to spca_df
spca_df = sample_coordinates
    
# Load the metadata from the subject-metadata.tsv file
metadata_df = pd.read_csv(f'../Data/metabolomics/ctf-results_Metabolomics_11182024_Method2/subject-metadata.tsv', sep='\t')
    
# Map the index of spca_df (sample names) to the SampleID in metadata_df to get the 'group'
spca_df["Group"] = spca_df.index.map(metadata_df.set_index("#SampleID")["group"])
    
# Rename the columns of spca_df for clarity (PC1, PC2, PC3)
spca_df.columns = ['PC1', 'PC2', 'PC3', 'Group']
    
# Display the updated spca_df
print(spca_df)
    
# Plotting
mm = 1/25.4
fig, ax = plt.subplots(1, 1, figsize=(90*mm, 110*mm))
    
# Create scatter plot with different colors based on the Group column
for group_name, group in spca_df.groupby('Group'):
    sns.scatterplot(
        data=group,
        x="PC1",
        y="PC2",
        facecolor=group_colors[group_name],  # Semi-transparent fill color
        edgecolor=group_colors[group_name],  # Colored border
        alpha=0.8,  # Transparency for the fill
        linewidth=0.5,  # Thickness of the borders
        ax=ax,
        label=group_name_mapping[group_name]
    )
    
    # Calculate mean and covariance matrix for the group
    points = group[["PC1", "PC2"]].values
    mean = np.mean(points, axis=0)
    cov = np.cov(points, rowvar=False)
    
    # Draw the ellipse for the group (with scale factor applied)
    draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
    # Add a subtle star (8-pointed) at the mean location for each group
    ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=100, edgecolor='black', zorder=5, linewidth=0.5)
    
# Add axis labels and include the explained variance percentages
ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=7)
ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=7)
    
# Simplify the ticks to avoid overcrowding
yticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_yticks(yticklabels)
ax.set_yticklabels(yticklabels, fontsize=7)
    
xticklabels = [-0.4, -0.2, 0.0, 0.2, 0.4, 0.6]
ax.set_xticks(xticklabels)
ax.set_xticklabels(xticklabels, fontsize=7)
    
# Define the title
plot_title = "CTF Analysis Metabolomics"

# Add the title consistently in the top-left corner using axes coordinates
ax.text(0.05, 1.10, plot_title, fontsize=8, va='top', ha='left', transform=ax.transAxes)

# Customize legend
plt.legend(
    frameon=False,
    fontsize=7,
    title_fontsize=7
)
    
# Adjust plot style and save
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
    
plt.tight_layout()
plt.savefig(f"../Figures/metabolomics_Figures/CTF_method2.png", dpi=600)
plt.show()


In [188]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu

# rclr transform the biom table
rclr_transformed_qza = rclr_transformation(biom_table)
rclr_transformed_df = pd.DataFrame(
    rclr_transformed_qza.matrix_data.toarray(),
    index=rclr_transformed_qza.ids(axis='observation'),
    columns=rclr_transformed_qza.ids(axis='sample')
)
rclr_transformed_df = rclr_transformed_df.fillna(0)

# Map metadata categories to rclr-transformed samples
metadata_df = pd.read_csv('/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Metadata/metadata_final_22102024.tsv', sep='\t')
metadata_df = metadata_df.set_index('SampleID')
rclr_transformed_df = rclr_transformed_df.T.join(metadata_df[['group']])
rclr_transformed_df = rclr_transformed_df.rename(columns={'group': 'category_label'})

# Filter features for top 20
top_features = filtered_features.nlargest(20, 'Magnitude')

# Merge `Compound_Name` and `mz` for labeling
top_features = top_features.reset_index().rename(columns={'index': 'FeatureID'})
top_features['label'] = top_features.apply(
    lambda row: f"{row['Compound_Name']} (mz: {row['mz']:.3f})" if pd.notna(row['Compound_Name']) else f"{row['FeatureID']} (mz: {row['mz']:.3f})", axis=1
)

# Set colors for categories
colors = {
    "Healthy": "#423fa6",
    "Acne_NL": "#67b2be",
    "Acne_L": "#df7963"
}

# Define custom significance label with p-value
def significance_label(p_value):
    if p_value < 0.001:
        return f"*** (p={p_value:.3e})"
    elif p_value < 0.01:
        return f"** (p={p_value:.3e})"
    elif p_value < 0.05:
        return f"* (p={p_value:.3e})"
    else:
        return f"ns (p={p_value:.3e})"

# Add significance brackets
def add_significance_bracket(ax, start, end, height, label):
    ax.plot([start, end], [height, height], color='black', lw=1)
    ax.text((start + end) / 2, height, label, ha='center', va='bottom', fontsize=8)

# Initialize variables for multi-plot layout
plots_per_page = 6
figures = []
current_fig, current_axes = None, []

# Loop through each feature in the top 20
for idx, row in top_features.iterrows():
    feature = row['FeatureID']
    label = row['label']

    # Create new figure if needed
    if idx % plots_per_page == 0:
        if current_fig:
            figures.append(current_fig)
        current_fig, current_axes = plt.subplots(
            nrows=2, ncols=3, figsize=(15, 10), constrained_layout=True
        )
        current_axes = current_axes.flatten()

    # Prepare data for the feature
    feature_data = rclr_transformed_df[[feature, 'category_label']].copy()
    feature_data = feature_data.rename(columns={feature: 'value'})
    ax = current_axes[idx % plots_per_page]

    # Create boxplot and scatter points
    sns.boxplot(
        x='category_label', y='value', data=feature_data, palette=colors, ax=ax,
        showfliers=False, width=0.5
    )
    sns.stripplot(
        x='category_label', y='value', data=feature_data, palette=colors,
        jitter=0.2, size=4, ax=ax
    )

    # Add labels and customize appearance
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('rclr Transformed Value', fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)  # Remove grid lines

    # Perform Mann-Whitney U tests and add significance brackets
    groups = ['Healthy', 'Acne_L', 'Acne_NL']
    max_height = feature_data['value'].max()
    comparisons = [(0, 1), (0, 2), (1, 2)]
    for i, (start_idx, end_idx) in enumerate(comparisons):
        group1 = groups[start_idx]
        group2 = groups[end_idx]
        data1 = feature_data[feature_data['category_label'] == group1]['value']
        data2 = feature_data[feature_data['category_label'] == group2]['value']
        _, p_value = mannwhitneyu(data1, data2)
        label = significance_label(p_value)

        # Add significance bracket
        height = max_height + (i + 1) * 0.1 * max_height
        add_significance_bracket(ax, start_idx, end_idx, height, label)

# Add last figure
if current_fig:
    figures.append(current_fig)

# Save figures
for page_idx, fig in enumerate(figures):
    fig_path = f"../Figures/metabolomics_Figures/CTF_compound_plots_page_{page_idx + 1}.png"
    fig.savefig(fig_path, dpi=300)
    plt.close(fig)
    print(f"Saved page: {fig_path}")


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2394409440.py:82: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2394409440.py:82: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2394409440.py:82: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2394409440.py:82: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2394409440.py:82: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78172/2394409440.py:82: FutureWarning: Passing `palette` without assigning `hue` 

Saved page: ../Figures/metabolomics_Figures/CTF_compound_plots_page_1.png
Saved page: ../Figures/metabolomics_Figures/CTF_compound_plots_page_2.png
Saved page: ../Figures/metabolomics_Figures/CTF_compound_plots_page_3.png
Saved page: ../Figures/metabolomics_Figures/CTF_compound_plots_page_4.png
